# Stage 2 — QOTPH + Adaptive LazyQ + Resource-Aware Federated Hybrid QML

This notebook extends the validated parallel data-reuploading QBank with:

- QOTPH Algorithm 1 as a protected-execution equivalence control.
- QOTPH Algorithm 2 as the final client-side output-decryption mode.
- An analytic Algorithm-2 training path for scalable development.
- A finite-shot count-returning Algorithm-2 path with explicit parameter-shift gradients.
- Adaptive LazyQ driven by quality, stagnation, uncertainty, drift, fairness debt, and resource capacity.
- Heterogeneous client epochs, rotating circuit masks, and optional sparse uploads.
- FedProx, quality-aware aggregation, layer sparing, validation-quality DP, optional update DP, and selective CKKS.

> **Scientific scope:** The finite-shot `qotph_counts_smoke` profile is the closest implementation to the design blueprint's Algorithm-2 training path. The faster `analytic` path uses the mathematically equivalent local Pauli-Z sign correction for development and ablation. Gate conventions and backend wire ordering must still be validated against the official QOTPH source before a paper-faithful claim.

> **Threat-model warning:** QOTPH masks the delegated quantum state and output, but the present variational integration does not by itself hide the magnitudes of client-generated rotation angles sent to the backend. The notebook logs this as allowed leakage; a separate private-angle encoding study is required before claiming angle confidentiality.


In [ ]:
!pip install tenseal syft pennylane
!pip install protobuf==3.20.3

In [1]:
import os
import math
import copy
import random
import pickle
import time
import hashlib
import secrets
import warnings
from collections import OrderedDict, defaultdict
from dataclasses import dataclass, asdict
from enum import Enum
from typing import List, Tuple, Dict, Optional, Callable, Union, cast, Any

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset

import torchvision
from torchvision import datasets, transforms

import pennylane as qml
import tenseal as ts
import syft as sy

from io import BytesIO
from functools import reduce
from logging import WARNING
from sklearn.metrics import roc_curve, auc, confusion_matrix


# Utility Functions


In [2]:
def choice_device(device):
    if torch.cuda.is_available() and device != "cpu":
        device = "cuda:0"
    elif (
        torch.backends.mps.is_available()
        and torch.backends.mps.is_built()
        and device != "cpu"
    ):
        device = "mps"
    else:
        device = "cpu"
    return device


def classes_string(name_dataset):
    if name_dataset == "cifar":
        return (
            "plane",
            "car",
            "bird",
            "cat",
            "deer",
            "dog",
            "frog",
            "horse",
            "ship",
            "truck",
        )
    elif name_dataset == "svhn":
        return tuple(str(i) for i in range(10))  # SVHN has 10 classes (digits 0-9)
    elif name_dataset == "caltech101":
        return tuple([f"class_{i}" for i in range(101)])  # Caltech101 has 101 classes
    elif name_dataset == "stanfordcars":
        return tuple([f"class_{i}" for i in range(196)])  # StanfordCars has 196 classes
    elif name_dataset == "fashion_mnist":
        return (
            "T-shirt/top",
            "Trouser",
            "Pullover",
            "Dress",
            "Coat",
            "Sandal",
            "Shirt",
            "Sneaker",
            "Bag",
            "Ankle boot",
        )
    else:
        raise ValueError(f"Unsupported dataset: {name_dataset}")


def save_matrix(y_true, y_pred, path, classes):
    y_true_mapped = [classes[label] for label in y_true]
    y_pred_mapped = [classes[label] for label in y_pred]
    cf_matrix_normalized = confusion_matrix(
        y_true_mapped, y_pred_mapped, labels=classes, normalize="all"
    )
    cf_matrix_round = np.round(cf_matrix_normalized, 2)
    df_cm = pd.DataFrame(
        cf_matrix_round, index=[i for i in classes], columns=[i for i in classes]
    )
    plt.figure(figsize=(12, 7))
    sn.heatmap(df_cm, annot=True)
    plt.xlabel("Predicted label", fontsize=13)
    plt.ylabel("True label", fontsize=13)
    plt.title("Confusion Matrix", fontsize=15)
    plt.savefig(path)
    plt.close()


def save_roc(targets, y_proba, path, nbr_classes):
    y_true = np.zeros(shape=(len(targets), nbr_classes))
    for i in range(len(targets)):
        y_true[i, targets[i]] = 1
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(nbr_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true.ravel(), y_proba.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(nbr_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(nbr_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= nbr_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
    plt.figure()
    plt.plot(
        fpr["micro"],
        tpr["micro"],
        label=f"micro-average ROC curve (area = {roc_auc['micro']:.2f})",
        color="deeppink",
        linestyle=":",
        linewidth=4,
    )
    plt.plot(
        fpr["macro"],
        tpr["macro"],
        label=f"macro-average ROC curve (area = {roc_auc['macro']:.2f})",
        color="navy",
        linestyle=":",
        linewidth=4,
    )
    lw = 2
    for i in range(nbr_classes):
        plt.plot(
            fpr[i],
            tpr[i],
            lw=lw,
            label=f"ROC curve of class {i} (area = {roc_auc[i]:.2f})",
        )
    plt.plot([0, 1], [0, 1], "k--", lw=lw, label="Worst case")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver operating characteristic (ROC) Curve OvR")
    plt.legend(loc="lower right")
    plt.savefig(path)
    plt.close()


def save_graphs(path_save, local_epoch, results, end_file=""):
    os.makedirs(path_save, exist_ok=True)
    print("Saving graphs in ", path_save)
    plot_graph(
        [[*range(local_epoch)]] * 2,
        [results["train_acc"], results["val_acc"]],
        "Epochs",
        "Accuracy (%)",
        ["Training accuracy", "Validation accuracy"],
        "Accuracy curves",
        path_save + "Accuracy_curves" + end_file,
    )
    plot_graph(
        [[*range(local_epoch)]] * 2,
        [results["train_loss"], results["val_loss"]],
        "Epochs",
        "Loss",
        ["Training loss", "Validation loss"],
        "Loss curves",
        path_save + "Loss_curves" + end_file,
    )


def plot_graph(
    list_xplot, list_yplot, x_label, y_label, curve_labels, title, path=None
):
    lw = 2
    plt.figure()
    for i in range(len(curve_labels)):
        plt.plot(list_xplot[i], list_yplot[i], lw=lw, label=curve_labels[i])
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    if curve_labels:
        plt.legend(loc="lower right")
    if path:
        plt.savefig(path)
    plt.close()


def get_parameters2(net, context_client=None) -> List[np.ndarray]:
    if context_client:
        encrypted_tensor = crypte(net.state_dict(), context_client)
        return [layer.get_weight() for layer in encrypted_tensor]
    return [val.cpu().numpy() for _, val in net.state_dict().items()]


def set_parameters(net, parameters: List[np.ndarray], context_client=None):
    state_dict = net.state_dict()
    params_dict = zip(state_dict.keys(), parameters)
    if context_client:
        secret_key = context_client.secret_key()
        dico = {k: deserialized_layer(k, v, context_client) for k, v in params_dict}
        new_state_dict = OrderedDict()
        for k, v in dico.items():
            if isinstance(v, CryptedLayer):
                decrypted = v.decrypt(secret_key)
                shape = state_dict[k].shape
                new_state_dict[k] = torch.Tensor(np.array(decrypted).reshape(shape))
            else:
                new_state_dict[k] = torch.Tensor(v.get_weight())
    else:
        new_state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
    net.load_state_dict(new_state_dict, strict=True)
    print("Updated model parameters")

# Security-related classes and functions


In [3]:
class Layer:
    def __init__(self, name_layer, weight):
        self.name = name_layer
        self.weight_array = weight

    def get_name(self):
        return self.name

    def get_weight(self):
        return self.weight_array

    def __add__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array + weights)

    def __sub__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array - weights)

    def __mul__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        return Layer(self.name, self.weight_array * weights)

    def __truediv__(self, other):
        weights = other.get_weight() if isinstance(other, Layer) else other
        weights = self.weight_array * (1 / weights)
        return Layer(self.name, weights)

    def __len__(self):
        somme = 1
        for elem in self.weight_array.shape:
            somme *= elem
        return somme

    def shape(self):
        return self.weight_array.shape

    def sum(self, axis=0):
        return Layer(f"sum_{self.name}", self.weight_array.sum(axis=axis))

    def mean(self, axis=0):
        weights = self.weight_array.sum(axis=axis) * (1 / self.weight_array.shape[axis])
        return Layer(f"sum_{self.name}", weights)

    def decrypt(self, sk=None):
        return self.weight_array.tolist()

    def serialize(self):
        return {self.name: self.weight_array}


class CryptedLayer(Layer):
    def __init__(self, name_layer, weight, contexte=None):
        super(CryptedLayer, self).__init__(name_layer, weight)
        if isinstance(weight, (ts.tensors.CKKSTensor, bytes)):
            self.weight_array = weight
        else:
            self.weight_array = ts.ckks_tensor(contexte, weight.cpu().detach().numpy())

    def __add__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array + weights)

    def __sub__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array - weights)

    def __mul__(self, other):
        weights = other.get_weight() if isinstance(other, CryptedLayer) else other
        return CryptedLayer(self.name, self.weight_array * weights)

    def __truediv__(self, other):
        try:
            weights = other.get_weight() if isinstance(other, CryptedLayer) else other
            weights = self.weight_array * (1 / weights)
        except:
            print("Error: division operator not supported by SEAL")
            weights = []
        return CryptedLayer(self.name, weights)

    def shape(self):
        return self.weight_array.shape

    def sum(self, axis=0):
        return CryptedLayer(f"sum_{self.name}", self.weight_array.sum(axis=axis))

    def mean(self, axis=0):
        weights = self.weight_array.sum(axis=axis) * (1 / self.weight_array.shape[axis])
        return CryptedLayer(f"sum_{self.name}", weights)

    def decrypt(self, sk=None):
        return (
            self.weight_array.decrypt(sk).tolist()
            if sk
            else self.weight_array.decrypt().tolist()
        )

    def serialize(self):
        return {self.name: self.weight_array.serialize()}


def context():
    cont = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60],
    )
    cont.generate_galois_keys()
    cont.global_scale = 2**40
    return cont


def crypte(client_w, context_c):
    encrypted = []
    for name_layer, weight_array in client_w.items():
        if name_layer in {"fc4.weight", "classifier.weight"}:
            encrypted.append(CryptedLayer(name_layer, weight_array, context_c))
        else:
            encrypted.append(Layer(name_layer, weight_array))
    return encrypted


def read_query(file_path):
    if os.path.exists(file_path):
        with open(file_path, "rb") as file:
            query_str = pickle.load(file)
        contexte = query_str["contexte"]
        del query_str["contexte"]
        return query_str, contexte
    else:
        print(f"File {file_path} does not exist")
        return None, None


def write_query(file_path, client_query):
    with open(file_path, "wb") as file:
        encode_str = pickle.dumps(client_query)
        file.write(encode_str)


def deserialized_layer(name_layer, weight_array, ctx):
    if isinstance(weight_array, bytes):
        return CryptedLayer(name_layer, ts.ckks_tensor_from(ctx, weight_array), ctx)
    elif isinstance(weight_array, ts.tensors.CKKSTensor):
        return CryptedLayer(name_layer, weight_array, ctx)
    else:
        return Layer(name_layer, weight_array)


# Data setup


In [4]:
NORMALIZE_DICT = {
    "cifar": dict(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    "svhn": dict(mean=(0.4377, 0.4438, 0.4728), std=(0.1980, 0.2010, 0.1970)),
    "caltech101": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    "stanfordcars": dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    "fashion_mnist": dict(mean=(0.2860,), std=(0.3530,)),
}


def _dataset_targets(dataset_obj) -> np.ndarray:
    """Return integer labels for torchvision-style datasets."""
    if hasattr(dataset_obj, "targets"):
        labels = dataset_obj.targets
    elif hasattr(dataset_obj, "labels"):
        labels = dataset_obj.labels
    else:
        raise AttributeError("Dataset does not expose .targets or .labels")
    if torch.is_tensor(labels):
        labels = labels.cpu().numpy()
    return np.asarray(labels, dtype=np.int64)


def iid_partition_indices(
    n_samples: int,
    num_clients: int,
    seed: int,
) -> List[np.ndarray]:
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n_samples)
    return [np.asarray(x, dtype=np.int64) for x in np.array_split(indices, num_clients)]




def balanced_static_full_partition_indices(
    labels: np.ndarray,
    num_clients: int,
    val_per_class_per_client: int,
    seed: int,
) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    """
    Split the complete training set once and keep client ownership fixed.

    Every class is divided as evenly as possible among all clients. A fixed
    balanced validation subset is reserved for every client before the
    remaining samples are assigned for training. For Fashion-MNIST with
    10 clients and 20 validation images/class/client, each client receives
    5,800 training and 200 validation images.
    """
    if num_clients <= 0:
        raise ValueError("num_clients must be positive")
    if val_per_class_per_client < 0:
        raise ValueError("val_per_class_per_client cannot be negative")

    rng = np.random.default_rng(seed)
    classes = np.unique(labels)
    train_indices: List[List[int]] = [[] for _ in range(num_clients)]
    val_indices: List[List[int]] = [[] for _ in range(num_clients)]

    for cls in classes:
        cls_indices = np.where(labels == cls)[0].astype(np.int64)
        rng.shuffle(cls_indices)

        required_val = num_clients * val_per_class_per_client
        if required_val >= len(cls_indices):
            raise ValueError(
                f"Class {cls} has {len(cls_indices)} samples, but "
                f"{required_val} are requested for client validation."
            )

        if required_val:
            cls_val = cls_indices[:required_val].reshape(
                num_clients,
                val_per_class_per_client,
            )
        else:
            cls_val = np.empty((num_clients, 0), dtype=np.int64)

        cls_train = cls_indices[required_val:]
        train_splits = np.array_split(cls_train, num_clients)

        for cid in range(num_clients):
            train_indices[cid].extend(train_splits[cid].tolist())
            val_indices[cid].extend(cls_val[cid].tolist())

    train_output, val_output = [], []
    for cid in range(num_clients):
        train_arr = np.asarray(train_indices[cid], dtype=np.int64)
        val_arr = np.asarray(val_indices[cid], dtype=np.int64)
        rng.shuffle(train_arr)
        rng.shuffle(val_arr)
        train_output.append(train_arr)
        val_output.append(val_arr)

    return train_output, val_output


def dirichlet_partition_indices(
    labels: np.ndarray,
    num_clients: int,
    alpha: float,
    seed: int,
    min_client_samples: int = 100,
    max_retries: int = 200,
) -> List[np.ndarray]:
    """
    Label-skew partition used in the AdeptHEQ-FL paper.
    A small alpha (0.1) creates strongly non-IID client label distributions.
    """
    if alpha <= 0:
        raise ValueError("Dirichlet alpha must be > 0")

    rng = np.random.default_rng(seed)
    classes = np.unique(labels)
    average_size = len(labels) / num_clients

    for _ in range(max_retries):
        client_indices: List[List[int]] = [[] for _ in range(num_clients)]

        for cls in classes:
            cls_indices = np.where(labels == cls)[0].copy()
            rng.shuffle(cls_indices)

            proportions = rng.dirichlet(np.full(num_clients, alpha))

            # Mild balancing prevents a few clients from absorbing almost all samples.
            balance_mask = np.asarray(
                [len(idx) < average_size for idx in client_indices],
                dtype=np.float64,
            )
            proportions = proportions * balance_mask
            if proportions.sum() == 0:
                proportions = np.ones(num_clients, dtype=np.float64)
            proportions = proportions / proportions.sum()

            split_points = (
                np.cumsum(proportions)[:-1] * len(cls_indices)
            ).astype(int)
            class_splits = np.split(cls_indices, split_points)

            for cid, split in enumerate(class_splits):
                client_indices[cid].extend(split.tolist())

        sizes = [len(idx) for idx in client_indices]
        if min(sizes) >= min_client_samples:
            output = []
            for idx in client_indices:
                arr = np.asarray(idx, dtype=np.int64)
                rng.shuffle(arr)
                output.append(arr)
            return output

    raise RuntimeError(
        f"Could not create a Dirichlet partition with at least "
        f"{min_client_samples} samples/client after {max_retries} retries. "
        "Reduce min_client_samples or increase alpha."
    )


def _split_client_train_val(
    indices: np.ndarray,
    val_fraction: float,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be between 0 and 1")
    rng = np.random.default_rng(seed)
    shuffled = np.asarray(indices, dtype=np.int64).copy()
    rng.shuffle(shuffled)

    n_val = max(1, int(round(len(shuffled) * val_fraction)))
    n_val = min(n_val, len(shuffled) - 1)
    return shuffled[n_val:], shuffled[:n_val]


def _make_dataset(
    dataset: str,
    root: str,
    train: bool,
    transformer,
):
    if dataset == "cifar":
        return datasets.CIFAR10(root + dataset, train=train, download=True, transform=transformer)
    if dataset == "svhn":
        split = "train" if train else "test"
        return datasets.SVHN(root + "svhn", split=split, download=True, transform=transformer)
    if dataset == "fashion_mnist":
        return datasets.FashionMNIST(
            root + "fashion_mnist",
            train=train,
            download=True,
            transform=transformer,
        )
    if dataset == "stanfordcars":
        split = "train" if train else "test"
        return datasets.StanfordCars(
            root + "stanfordcars",
            split=split,
            download=True,
            transform=transformer,
        )
    if dataset == "caltech101":
        split = "train" if train else "test"
        return datasets.ImageFolder(root + f"caltech101/{split}", transform=transformer)
    raise ValueError(f"Unsupported dataset: {dataset}")


def load_datasets(
    num_clients: int,
    batch_size: int,
    resize: Optional[int],
    seed: int,
    num_workers: int,
    splitter: float = 10.0,
    dataset: str = "fashion_mnist",
    data_path: str = "./data/",
    partition_mode: str = "dirichlet",
    dirichlet_alpha: float = 0.1,
    min_client_samples: int = 100,
    balanced_val_per_class_per_client: int = 20,
):
    """
    Returns:
        trainloaders, valloaders, testloader, partition_stats

    splitter is the local validation percentage, kept for compatibility
    with the original notebook.
    """
    list_transforms = [
        transforms.ToTensor(),
        transforms.Normalize(**NORMALIZE_DICT[dataset]),
    ]

    if dataset in ["caltech101", "stanfordcars"] and resize is not None:
        list_transforms = [transforms.Resize((resize, resize))] + list_transforms
    elif dataset == "svhn":
        list_transforms = [transforms.Resize((32, 32))] + list_transforms

    transformer = transforms.Compose(list_transforms)
    trainset = _make_dataset(dataset, data_path, train=True, transformer=transformer)
    testset = _make_dataset(dataset, data_path, train=False, transformer=transformer)

    labels = _dataset_targets(trainset)
    static_val_indices = None

    if partition_mode == "dirichlet":
        client_indices = dirichlet_partition_indices(
            labels=labels,
            num_clients=num_clients,
            alpha=dirichlet_alpha,
            seed=seed,
            min_client_samples=min_client_samples,
        )
    elif partition_mode == "iid":
        client_indices = iid_partition_indices(len(trainset), num_clients, seed)
    elif partition_mode == "balanced_static_full":
        client_indices, static_val_indices = (
            balanced_static_full_partition_indices(
                labels=labels,
                num_clients=num_clients,
                val_per_class_per_client=(
                    balanced_val_per_class_per_client
                ),
                seed=seed,
            )
        )
    else:
        raise ValueError(
            "partition_mode must be 'dirichlet', 'iid', "
            "or 'balanced_static_full'"
        )

    trainloaders, valloaders = [], []
    partition_stats = []

    for cid, indices in enumerate(client_indices):
        if static_val_indices is None:
            train_idx, val_idx = _split_client_train_val(
                indices,
                val_fraction=splitter / 100.0,
                seed=seed + 10_000 + cid,
            )
            total_indices = indices
        else:
            train_idx = np.asarray(indices, dtype=np.int64)
            val_idx = np.asarray(static_val_indices[cid], dtype=np.int64)
            total_indices = np.concatenate([train_idx, val_idx])

        generator = torch.Generator().manual_seed(seed + cid)
        trainloaders.append(
            DataLoader(
                Subset(trainset, train_idx.tolist()),
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=torch.cuda.is_available(),
                generator=generator,
            )
        )
        valloaders.append(
            DataLoader(
                Subset(trainset, val_idx.tolist()),
                batch_size=batch_size,
                shuffle=False,
                num_workers=num_workers,
                pin_memory=torch.cuda.is_available(),
            )
        )

        class_counts = np.bincount(
            labels[total_indices],
            minlength=len(np.unique(labels)),
        )
        partition_stats.append(
            {
                "client": cid,
                "total": int(len(total_indices)),
                "train": int(len(train_idx)),
                "val": int(len(val_idx)),
                "class_counts": class_counts.tolist(),
            }
        )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    print(f"Partition mode: {partition_mode}")
    if partition_mode == "dirichlet":
        print(f"Dirichlet alpha: {dirichlet_alpha}")
    elif partition_mode == "balanced_static_full":
        print(
            "Static full-data allocation: every client retains a fixed, "
            "class-balanced local dataset."
        )
    for stat in partition_stats:
        print(
            f"Client {stat['client']:02d} | total={stat['total']:5d} | "
            f"class_counts={stat['class_counts']}"
        )

    return trainloaders, valloaders, testloader, partition_stats

# ============================================================
# Balanced rotating per-round client data
# ============================================================

class BalancedRotatingClientScheduler:
    """
    Creates a class-balanced local dataset for every participating client
    in every FL round.

    Guarantees in reuse_scope="per_client":
      1. Every client receives exactly samples_per_class_per_client
         samples from every class in each round.
      2. Clients do not share a training block within the same round.
      3. The same client does not receive the same training block again
         until all available class blocks have been cycled through.
      4. A block may be reused by a different client in a later round.

    In reuse_scope="global":
      No training block is reused by any client across rounds, but the
      maximum number of rounds is much smaller.
    """

    def __init__(
        self,
        trainset,
        num_clients: int,
        batch_size: int,
        samples_per_class_per_client: int = 100,
        val_per_class_per_client: int = 20,
        seed: int = 42,
        num_workers: int = 0,
        reuse_scope: str = "per_client",
    ):
        self.trainset = trainset
        self.num_clients = int(num_clients)
        self.batch_size = int(batch_size)
        self.samples_per_class_per_client = int(
            samples_per_class_per_client
        )
        self.val_per_class_per_client = int(
            val_per_class_per_client
        )
        self.seed = int(seed)
        self.num_workers = int(num_workers)
        self.reuse_scope = reuse_scope.lower()

        if self.samples_per_class_per_client <= 0:
            raise ValueError(
                "samples_per_class_per_client must be positive"
            )
        if self.val_per_class_per_client < 0:
            raise ValueError(
                "val_per_class_per_client cannot be negative"
            )
        if self.reuse_scope not in {"per_client", "global"}:
            raise ValueError(
                "reuse_scope must be 'per_client' or 'global'"
            )

        labels = _dataset_targets(trainset)
        self.labels = labels
        self.classes = np.unique(labels).astype(int).tolist()
        self.num_classes = len(self.classes)

        rng = np.random.default_rng(self.seed)
        self.class_blocks = {}
        self.validation_indices = {
            cid: []
            for cid in range(self.num_clients)
        }

        block_counts = []

        for cls in self.classes:
            cls_indices = np.where(labels == cls)[0].astype(
                np.int64
            )
            rng.shuffle(cls_indices)

            required_val = (
                self.num_clients
                * self.val_per_class_per_client
            )
            if required_val >= len(cls_indices):
                raise ValueError(
                    f"Class {cls} has only {len(cls_indices)} samples, "
                    f"but {required_val} were requested for validation."
                )

            validation_pool = cls_indices[:required_val]
            training_pool = cls_indices[required_val:]

            for cid in range(self.num_clients):
                start = cid * self.val_per_class_per_client
                stop = start + self.val_per_class_per_client
                self.validation_indices[cid].extend(
                    validation_pool[start:stop].tolist()
                )

            n_blocks = (
                len(training_pool)
                // self.samples_per_class_per_client
            )
            if n_blocks < self.num_clients:
                raise ValueError(
                    f"Not enough class-{cls} blocks for "
                    f"{self.num_clients} clients."
                )

            usable = (
                n_blocks
                * self.samples_per_class_per_client
            )
            training_pool = training_pool[:usable]

            self.class_blocks[cls] = [
                training_pool[
                    block_id
                    * self.samples_per_class_per_client:
                    (block_id + 1)
                    * self.samples_per_class_per_client
                ].copy()
                for block_id in range(n_blocks)
            ]
            block_counts.append(n_blocks)

        self.num_blocks = min(block_counts)

        # Choose a stride that is coprime to num_blocks. This ensures
        # that a fixed client cycles through all blocks before repeating.
        stride = self.num_clients + 1
        while math.gcd(stride, self.num_blocks) != 1:
            stride += 1
        self.block_stride = stride

        self.max_global_nonrepeat_rounds = (
            self.num_blocks // self.num_clients
        )
        self.max_per_client_nonrepeat_rounds = (
            self.num_blocks
        )

        self.valloaders = []
        for cid in range(self.num_clients):
            val_indices = np.asarray(
                self.validation_indices[cid],
                dtype=np.int64,
            )
            val_rng = np.random.default_rng(
                self.seed + 50_000 + cid
            )
            val_rng.shuffle(val_indices)

            self.valloaders.append(
                DataLoader(
                    Subset(
                        self.trainset,
                        val_indices.tolist(),
                    ),
                    batch_size=self.batch_size,
                    shuffle=False,
                    num_workers=self.num_workers,
                    pin_memory=torch.cuda.is_available(),
                )
            )

    def _block_id(
        self,
        round_index: int,
        cid: int,
    ) -> int:
        if self.reuse_scope == "global":
            block_id = (
                round_index * self.num_clients + cid
            )
            if block_id >= self.num_blocks:
                raise RuntimeError(
                    "Strict global non-reuse has exhausted the "
                    "available class blocks. With the current settings, "
                    f"the maximum is {self.max_global_nonrepeat_rounds} "
                    "rounds. Reduce samples per class per client, reduce "
                    "the number of clients, or use reuse_scope='per_client'."
                )
            return block_id

        # Per-client non-repeat schedule. Same-round clients use
        # distinct blocks. A client does not repeat until num_blocks rounds.
        return (
            cid + round_index * self.block_stride
        ) % self.num_blocks

    def get_round_loaders(
        self,
        round_index: int,
        selected_clients: List[int],
    ):
        loaders = {}
        stats = {}

        for cid in selected_clients:
            block_id = self._block_id(
                round_index=round_index,
                cid=cid,
            )

            indices = []
            for cls in self.classes:
                indices.extend(
                    self.class_blocks[cls][block_id].tolist()
                )

            indices = np.asarray(
                indices,
                dtype=np.int64,
            )
            round_rng = np.random.default_rng(
                self.seed
                + 100_000
                + 10_000 * round_index
                + cid
            )
            round_rng.shuffle(indices)

            generator = torch.Generator().manual_seed(
                self.seed
                + 200_000
                + 10_000 * round_index
                + cid
            )

            loaders[cid] = DataLoader(
                Subset(
                    self.trainset,
                    indices.tolist(),
                ),
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=self.num_workers,
                pin_memory=torch.cuda.is_available(),
                generator=generator,
            )

            class_counts = np.bincount(
                self.labels[indices],
                minlength=self.num_classes,
            )
            stats[cid] = {
                "client": int(cid),
                "round": int(round_index + 1),
                "block_id": int(block_id),
                "total": int(len(indices)),
                "class_counts": class_counts.tolist(),
            }

        return loaders, stats


def load_balanced_rotating_datasets(
    num_clients: int,
    batch_size: int,
    seed: int,
    num_workers: int,
    dataset: str = "fashion_mnist",
    data_path: str = "./data/",
    samples_per_class_per_client: int = 100,
    val_per_class_per_client: int = 20,
    reuse_scope: str = "per_client",
):
    """
    Builds the rotating balanced scheduler, fixed balanced local
    validation loaders, and the standard global test loader.
    """
    transformer = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(
                **NORMALIZE_DICT[dataset]
            ),
        ]
    )

    trainset = _make_dataset(
        dataset,
        data_path,
        train=True,
        transformer=transformer,
    )
    testset = _make_dataset(
        dataset,
        data_path,
        train=False,
        transformer=transformer,
    )

    scheduler = BalancedRotatingClientScheduler(
        trainset=trainset,
        num_clients=num_clients,
        batch_size=batch_size,
        samples_per_class_per_client=(
            samples_per_class_per_client
        ),
        val_per_class_per_client=(
            val_per_class_per_client
        ),
        seed=seed,
        num_workers=num_workers,
        reuse_scope=reuse_scope,
    )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    print("Partition mode: balanced_rotating")
    print(
        "Training samples/client/round: "
        f"{samples_per_class_per_client} per class "
        f"({samples_per_class_per_client * len(scheduler.classes)} total)"
    )
    print(
        "Validation samples/client: "
        f"{val_per_class_per_client} per class "
        f"({val_per_class_per_client * len(scheduler.classes)} total)"
    )
    print(f"Reuse scope: {reuse_scope}")
    print(f"Class blocks available: {scheduler.num_blocks}")
    print(f"Rotation stride: {scheduler.block_stride}")
    print(
        "Maximum no-repeat rounds for each client: "
        f"{scheduler.max_per_client_nonrepeat_rounds}"
    )
    print(
        "Maximum strict globally non-repeating rounds: "
        f"{scheduler.max_global_nonrepeat_rounds}"
    )

    validation_stats = []
    for cid, loader in enumerate(scheduler.valloaders):
        indices = np.asarray(
            loader.dataset.indices,
            dtype=np.int64,
        )
        counts = np.bincount(
            scheduler.labels[indices],
            minlength=len(scheduler.classes),
        )
        validation_stats.append(
            {
                "client": cid,
                "total": int(len(indices)),
                "class_counts": counts.tolist(),
            }
        )

    return (
        scheduler,
        scheduler.valloaders,
        testloader,
        validation_stats,
    )


# Training and testing functions


In [5]:
def test(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: Union[torch.nn.Module, Tuple],
    device: torch.device,
):
    model.eval()
    test_loss, test_acc = 0, 0
    y_pred = []
    y_true = []
    y_proba = []
    softmax = nn.Softmax(dim=1)
    with torch.inference_mode():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            probas_output = softmax(output)
            y_proba.extend(probas_output.detach().cpu().numpy())
            loss = loss_fn(output, labels)
            test_loss += loss.item()
            labels = labels.data.cpu().numpy()
            y_true.extend(labels)
            preds = np.argmax(output.detach().cpu().numpy(), axis=1)
            y_pred.extend(preds)
            acc = (preds == labels).mean()
            test_acc += acc
    y_proba = np.array(y_proba)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc * 100, y_pred, y_true, y_proba


def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: Union[torch.nn.Module, Tuple],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, labels)
        train_loss += loss.item()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(output, dim=1), dim=1)
        train_acc += (y_pred_class == labels).sum().item() / len(output)
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc * 100


def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: Union[torch.nn.Module, Tuple],
    epochs: int,
    device: torch.device,
) -> Dict[str, List]:
    results = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        train_loss, train_acc = train_step(
            model, train_dataloader, loss_fn, optimizer, device
        )
        val_loss, val_acc, *_ = test(model, test_dataloader, loss_fn, device)
        print(
            f"\tTrain Epoch: {epoch + 1} \tTrain_loss: {train_loss:.4f} | Train_acc: {train_acc:.4f} % | "
            f"Validation_loss: {val_loss:.4f} | Validation_acc: {val_acc:.4f} %"
        )
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)
    return results


def serialize_ndarray(ndarray):
    if isinstance(ndarray, ts.tensors.CKKSTensor):
        return ndarray.serialize()
    elif isinstance(ndarray, torch.Tensor):
        return serialize_ndarray(ndarray.cpu().detach().numpy())
    else:
        bytes_io = BytesIO()
        np.save(bytes_io, ndarray, allow_pickle=False)
        return bytes_io.getvalue()


def deserialize_ndarray(tensor, context):
    try:
        return ts.ckks_tensor_from(context, tensor)
    except:
        bytes_io = BytesIO(tensor)
        return np.load(bytes_io, allow_pickle=False)


def serialize_parameters(parameters):
    return [serialize_ndarray(param) for param in parameters]


def deserialize_parameters(serialized_params, context):
    return [deserialize_ndarray(param, context) for param in serialized_params]


def privatize_accuracy(true_acc: float, N: int, ε=1.0):
    sensitivity = 1.0 / N
    noise = np.random.laplace(0, sensitivity / ε)
    return np.clip(true_acc + noise, 0, 1)


def accuracy_weights(accuracies: List[float], τ=0.5) -> List[float]:
    scaled_acc = [a / τ for a in accuracies]
    max_scaled = max(scaled_acc)
    exp_acc = [np.exp(a - max_scaled) for a in scaled_acc]
    sum_exp = sum(exp_acc)
    return [e / sum_exp for e in exp_acc]


def compute_difference_norm(new_param, prev_param, context):
    if isinstance(new_param, ts.tensors.CKKSTensor):
        new_dec = new_param.decrypt(context.secret_key()).tolist()
        prev_dec = prev_param.decrypt(context.secret_key()).tolist()
        diff = np.array(new_dec) - np.array(prev_dec)
    else:
        if isinstance(new_param, torch.Tensor):
            new_param = new_param.cpu().numpy()
        if isinstance(prev_param, torch.Tensor):
            prev_param = prev_param.cpu().numpy()
        diff = new_param - prev_param
    return np.linalg.norm(diff)


def aggregate_serialized(results, context, τ=0.5):
    accuracies = [dp_acc for _, dp_acc in results]
    weights = accuracy_weights(accuracies, τ)

    weights_results = [
        (deserialize_parameters(serialized_params, context), w)
        for (serialized_params, _), w in zip(results, weights)
    ]

    aggregated_params = []
    for layer_idx in range(len(weights_results[0][0])):
        layer_updates = [weights[layer_idx] for weights, _ in weights_results]
        if isinstance(layer_updates[0], ts.tensors.CKKSTensor):
            weighted_sum = sum([layer * w for layer, w in zip(layer_updates, weights)])
        else:
            weighted_sum = sum([layer * w for layer, w in zip(layer_updates, weights)])
        aggregated_params.append(weighted_sum)
    return serialize_parameters(aggregated_params)

# Stage-2 configuration

Choose one `RUN_PROFILE`:

- `stage2_smoke`: fast end-to-end Algorithm-2 analytic check.
- `qotph_counts_smoke`: finite-shot encrypted counts with local decryption and parameter-shift gradients.
- `full_research`: complete proposed stack; computationally extreme.
- `manual`: edit every setting directly.


In [17]:

# ============================================================
# Stage-2 experiment configuration
# ============================================================
#
# RUN_PROFILE options:
#   "stage2_smoke"          -> fast Algorithm-2 analytic pipeline check
#   "qotph_counts_smoke"    -> paper-aligned finite-shot counts + parameter-shift
#   "full_research"         -> complete proposed architecture; extremely expensive
#   "manual"                -> use the values below exactly
#
RUN_PROFILE = "stage2_learning_smoke"

# Base environment.
data_path = "data/"
dataset = "fashion_mnist"
seed = 42
num_workers = 0
batch_size = 32
splitter = 10.0
device = "gpu"
number_clients = 3
save_results = "results/FL/"

# Learning budget.
rounds = 5
max_epochs = 3
frac_fit = 1.0
min_fit_clients = number_clients

# Data allocation:
#   "balanced_rotating", "balanced_static_full", "dirichlet", or "iid".
PARTITION_MODE = "balanced_rotating"
DIRICHLET_ALPHA = 0.5
MIN_CLIENT_SAMPLES = 100
BALANCED_SAMPLES_PER_CLASS_PER_CLIENT = 100
BALANCED_VAL_PER_CLASS_PER_CLIENT = 20
BALANCED_REUSE_SCOPE = "per_client"

# Hybrid architecture.
MODEL_VARIANT = "parallel_qbank"
LATENT_DIM = 16
N_QUBITS = 4
K_CIRCUITS = 4
R_UNIQUE = 2
RU_LAYERS = 2
LR_CLASSICAL = 1e-3
LR_QUANTUM = 5e-2

# ============================================================
# QOTPH protected delegated execution
# ============================================================
# "off", "algorithm1", or "algorithm2".
#
# Algorithm 1 decrypts inside the delegated circuit before measurement.
# It is included as a correctness/equivalence control.
#
# Algorithm 2 keeps the output encrypted at the backend. In analytic
# training mode, encrypted Pauli-Z expectations are corrected locally.
# In finite_shot_parameter_shift mode, encrypted counts are returned,
# decrypted locally, and differentiated with explicit parameter shift.
QOTPH_MODE = "algorithm2"

# "analytic" or "finite_shot_parameter_shift".
QOTPH_EXECUTION = "analytic"
QOTPH_SHOTS = 4096
QOTPH_TEST_SHOTS = 2048
QOTPH_KEY_SEED = 20260730
# "deterministic_research" gives paired, reproducible ablations.
# "secure_random" uses fresh secrets-based one-time-pad keys and is
# required for the final privacy run; never claim cryptographic privacy
# from deterministic keys whose seed is known.
QOTPH_KEY_MODE = "deterministic_research"
QOTPH_WIRE_ORDER = tuple(range(N_QUBITS))
RUN_QOTPH_UNIT_TESTS = True

# Finite-shot parameter-shift is intentionally guarded because its cost
# grows approximately as 2 * parameters * circuits * shots per fresh step.
ALLOW_EXPENSIVE_COUNTS_TRAINING = False
QOTPH_COUNTS_MAX_BATCH = 4

# ============================================================
# Fixed and adaptive quantum-backward scheduling
# ============================================================
# "naive", "lazy", or "adaptive_lazy".
QUANTUM_TRAINING = "naive"

# Fixed LazyQ settings.
TAU_DRIFT = 0.05
REFRESH_EVERY = 5
BETA_QGRAD = 0.5
GAMMA_STALE = 0.9

# Adaptive LazyQ safety and allocation.
WARMUP_FRESH_STEPS = 8
MIN_REFRESH_RATIO = 0.05
MAX_REFRESH_RATIO = 0.50
MAX_STALE_STEPS = 60

ADAPT_ERROR_WEIGHT = 0.35
ADAPT_STAGNATION_WEIGHT = 0.15
ADAPT_DRIFT_WEIGHT = 0.30
ADAPT_UNCERTAINTY_WEIGHT = 0.10
ADAPT_FAIRNESS_WEIGHT = 0.10
ADAPT_QUALITY_RHO = 0.8
ADAPT_IMPROVEMENT_RHO = 0.8
ADAPT_UNCERTAINTY_RHO = 0.8
ADAPT_FAIRNESS_RHO = 0.8

# ============================================================
# Heterogeneous client resources
# ============================================================
USE_RESOURCE_AWARE_POLICY = False
USE_CIRCUIT_MASKING = False
USE_BANDWIDTH_SPARSE_UPLOAD = False
RESOURCE_PROFILE_SEED = 314159
BANDWIDTH_WARMUP_ROUNDS = 2
CIRCUIT_DROPOUT_PROB = 0.0

# ============================================================
# Federated optimization and communication
# ============================================================
AGGREGATION_MODE = "uniform"  # uniform, sample_size, quality, hybrid
TAU_AGG = 0.5
USE_QUALITY_EMA = False
QUALITY_EMA_RHO = 0.8

USE_FEDPROX = False
FEDPROX_MU = 1e-3

USE_SERVER_EMA = False
SERVER_EMA_BETA_CLASSICAL = 0.5
SERVER_EMA_BETA_QUANTUM = 0.0

USE_LAYER_SPARING = False
IMPORTANCE_EMA_ALPHA = 0.9
FREEZE_THRESHOLD = 0.001
FREEZE_WARMUP_ROUNDS = 3
FREEZE_PATIENCE = 2

# ============================================================
# Privacy at the federated boundary
# ============================================================
# Validation-quality DP, matching the AdeptHEQ-style pathway.
use_dp = False
DP_EPSILON = 1.0

# Optional experimental update-level Gaussian DP.
USE_UPDATE_DP = False
UPDATE_DP_CLIP_NORM = 1.0
UPDATE_DP_NOISE_MULTIPLIER = 0.0

# Selective CKKS encryption of the final classifier weight.
he = False
path_public_key = "server_key.pkl"
secret_path = "secret.pkl"

# ============================================================
# Run-profile overrides
# ============================================================
if RUN_PROFILE == "stage2_smoke":
    number_clients = 3
    rounds = 2
    max_epochs = 1
    frac_fit = 1.0
    min_fit_clients = 3
    batch_size = 16
    PARTITION_MODE = "balanced_rotating"
    BALANCED_SAMPLES_PER_CLASS_PER_CLIENT = 20
    BALANCED_VAL_PER_CLASS_PER_CLIENT = 10
    QOTPH_MODE = "algorithm2"
    QOTPH_EXECUTION = "analytic"
    QUANTUM_TRAINING = "adaptive_lazy"
    USE_RESOURCE_AWARE_POLICY = True
    USE_CIRCUIT_MASKING = True
    USE_FEDPROX = True
    AGGREGATION_MODE = "hybrid"
    use_dp = False
    he = False
    USE_LAYER_SPARING = False

elif RUN_PROFILE == "qotph_counts_smoke":
    number_clients = 1
    rounds = 1
    max_epochs = 1
    frac_fit = 1.0
    min_fit_clients = 1
    batch_size = 2
    PARTITION_MODE = "balanced_rotating"
    BALANCED_SAMPLES_PER_CLASS_PER_CLIENT = 2
    BALANCED_VAL_PER_CLASS_PER_CLIENT = 2
    QOTPH_MODE = "algorithm2"
    QOTPH_EXECUTION = "finite_shot_parameter_shift"
    QOTPH_SHOTS = 256
    QOTPH_TEST_SHOTS = 512
    ALLOW_EXPENSIVE_COUNTS_TRAINING = True
    QOTPH_COUNTS_MAX_BATCH = 2
    QUANTUM_TRAINING = "adaptive_lazy"
    WARMUP_FRESH_STEPS = 1
    MIN_REFRESH_RATIO = 0.05
    MAX_REFRESH_RATIO = 0.25
    USE_RESOURCE_AWARE_POLICY = False
    USE_CIRCUIT_MASKING = False
    USE_FEDPROX = False
    AGGREGATION_MODE = "uniform"
    use_dp = False
    he = False

elif RUN_PROFILE == "stage2_learning_smoke":

    number_clients = 3
    rounds = 5
    max_epochs = 3

    BALANCED_SAMPLES_PER_CLASS_PER_CLIENT = 50
    BALANCED_VAL_PER_CLASS_PER_CLIENT = 20

    QOTPH_MODE = "algorithm2"
    QOTPH_EXECUTION = "analytic"

    QUANTUM_TRAINING = "naive"
    # Or fixed LazyQ only after naive works.

    USE_RESOURCE_AWARE_POLICY = False
    USE_CIRCUIT_MASKING = False

    AGGREGATION_MODE = "uniform"

    USE_FEDPROX = False
    USE_QUALITY_EMA = False
    USE_LAYER_SPARING = False

    use_dp = False
    he = False

elif RUN_PROFILE == "full_research":
    number_clients = 10
    rounds = 20
    max_epochs = 3
    frac_fit = 0.5
    min_fit_clients = 5
    batch_size = 32
    PARTITION_MODE = "dirichlet"
    DIRICHLET_ALPHA = 0.5
    QOTPH_MODE = "algorithm2"
    QOTPH_EXECUTION = "finite_shot_parameter_shift"
    QOTPH_SHOTS = 4096
    QOTPH_KEY_MODE = "secure_random"
    ALLOW_EXPENSIVE_COUNTS_TRAINING = True
    QUANTUM_TRAINING = "adaptive_lazy"
    USE_RESOURCE_AWARE_POLICY = True
    USE_CIRCUIT_MASKING = True
    USE_BANDWIDTH_SPARSE_UPLOAD = True
    USE_FEDPROX = True
    AGGREGATION_MODE = "hybrid"
    USE_QUALITY_EMA = True
    USE_LAYER_SPARING = True
    use_dp = True
    he = True

elif RUN_PROFILE != "manual":
    raise ValueError(
        "RUN_PROFILE must be 'stage2_smoke', 'qotph_counts_smoke', "
        "'full_research', or 'manual'."
    )

min_fit_clients = min(min_fit_clients, number_clients)

if QOTPH_MODE not in {"off", "algorithm1", "algorithm2"}:
    raise ValueError("QOTPH_MODE must be 'off', 'algorithm1', or 'algorithm2'")

if QOTPH_KEY_MODE not in {"deterministic_research", "secure_random"}:
    raise ValueError(
        "QOTPH_KEY_MODE must be deterministic_research or secure_random"
    )

if QOTPH_EXECUTION not in {"analytic", "finite_shot_parameter_shift"}:
    raise ValueError(
        "QOTPH_EXECUTION must be 'analytic' or "
        "'finite_shot_parameter_shift'"
    )

if QUANTUM_TRAINING not in {"naive", "lazy", "adaptive_lazy"}:
    raise ValueError(
        "QUANTUM_TRAINING must be 'naive', 'lazy', or 'adaptive_lazy'"
    )

if QOTPH_MODE == "off" and QOTPH_EXECUTION == "finite_shot_parameter_shift":
    raise ValueError(
        "finite_shot_parameter_shift is only meaningful with QOTPH enabled"
    )

ENCRYPTED_FINAL_WEIGHT = (
    "fc4.weight"
    if MODEL_VARIANT == "original"
    else "classifier.weight"
)

qotph_tag = (
    "plain"
    if QOTPH_MODE == "off"
    else f"{QOTPH_MODE}_{QOTPH_EXECUTION}"
)

privacy_tag = (
    f"qdp{int(use_dp)}"
    f"_udp{int(USE_UPDATE_DP)}"
    f"_he{int(he)}"
)

RUN_NAME = (
    f"{MODEL_VARIANT}"
    f"_{PARTITION_MODE}"
    f"_{qotph_tag}"
    f"_{QUANTUM_TRAINING}"
    f"_resource{int(USE_RESOURCE_AWARE_POLICY)}"
    f"_{privacy_tag}"
    f"_R{rounds}"
    f"_E{max_epochs}"
    f"_seed{seed}"
)

model_save = os.path.join(
    save_results,
    f"{RUN_NAME}.pt",
)

DEVICE = torch.device(choice_device(device))
CLASSES = classes_string(dataset)

print(f"Run profile: {RUN_PROFILE}")
print(f"Run name:    {RUN_NAME}")
print(f"Device:      {DEVICE}")


Run profile: stage2_learning_smoke
Run name:    parallel_qbank_balanced_rotating_algorithm2_analytic_naive_resource0_qdp0_udp0_he0_R5_E3_seed42
Device:      cuda:0


# QOTPH protected quantum execution

Algorithm 1 and Algorithm 2 are implemented as **alternative modes**, not sequential layers.

- **Algorithm 1:** QOTP encryption, protected gate evaluation, in-circuit decryption, then measurement. Use it as a fidelity/equivalence control.
- **Algorithm 2:** QOTP encryption, protected gate evaluation, encrypted output return, and client-side reconstruction. Use it as the final delegated-quantum privacy mode.

For the restricted QBank gate set:

\[
R_Y(\theta) \rightarrow R_Y((-1)^{a\oplus b}\theta)
\]

and for `CNOT(c,t)`:

\[
a_t \leftarrow a_t\oplus a_c,\qquad b_c \leftarrow b_c\oplus b_t.
\]

The finite-shot mode reconstructs:

\[
\langle Z_j\rangle = \sum_y p(y)(-1)^{y_j}
\]

after decrypting each returned computational-basis bitstring locally.


In [18]:

# ============================================================
# QOTPH core: keys, gate schedule, Algorithm 1/2 executors,
# finite-shot local decryption, and protected parameter shift
# ============================================================

class QOTPHMode(str, Enum):
    OFF = "off"
    ALGORITHM_1 = "algorithm1"
    ALGORITHM_2 = "algorithm2"


@dataclass
class QOTPKeys:
    x: Tuple[int, ...]
    z: Tuple[int, ...]

    def copy(self) -> "QOTPKeys":
        return QOTPKeys(tuple(self.x), tuple(self.z))

    @classmethod
    def secure_random(cls, n_qubits: int) -> "QOTPKeys":
        return cls(
            x=tuple(secrets.randbits(1) for _ in range(n_qubits)),
            z=tuple(secrets.randbits(1) for _ in range(n_qubits)),
        )


def _stable_seed(*parts) -> int:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return int.from_bytes(digest[:8], "little") % (2**32)


def deterministic_qotp_keys(
    key_seed: int,
    client_id: int,
    round_number: int,
    invocation_id: int,
    circuit_id: int,
    n_qubits: int,
) -> QOTPKeys:
    rng = np.random.default_rng(
        _stable_seed(
            key_seed,
            client_id,
            round_number,
            invocation_id,
            circuit_id,
        )
    )
    return QOTPKeys(
        x=tuple(int(v) for v in rng.integers(0, 2, size=n_qubits)),
        z=tuple(int(v) for v in rng.integers(0, 2, size=n_qubits)),
    )


def qotph_key_schedule(
    initial_keys: QOTPKeys,
    n_qubits: int,
    reupload_layers: int,
):
    """
    Build the protected-RY sign schedule and propagate QOTP keys through
    every ring CNOT.

    For an encrypted state X^a Z^b |psi>, the restricted QBank gate rule is
        RY(theta) -> RY((-1)^(a XOR b) theta).

    CNOT(c, t) key propagation:
        a_t <- a_t XOR a_c
        b_c <- b_c XOR b_t
    """
    x_keys = list(initial_keys.x)
    z_keys = list(initial_keys.z)
    layer_signs = []

    for _ in range(reupload_layers):
        layer_signs.append(
            [
                -1.0 if (x_keys[wire] ^ z_keys[wire]) else 1.0
                for wire in range(n_qubits)
            ]
        )

        for control in range(n_qubits):
            target = (control + 1) % n_qubits
            old_x_control = x_keys[control]
            old_z_target = z_keys[target]
            x_keys[target] ^= old_x_control
            z_keys[control] ^= old_z_target

    return (
        np.asarray(layer_signs, dtype=np.float64),
        QOTPKeys(tuple(x_keys), tuple(z_keys)),
    )


def _apply_qotp_encryption(
    initial_x,
    initial_z,
    n_qubits: int,
):
    # Encryption is X^a Z^b |psi>; circuit order is Z then X.
    for wire in range(n_qubits):
        qml.RZ(math.pi * initial_z[wire], wires=wire)
        qml.RX(math.pi * initial_x[wire], wires=wire)


def _apply_qotp_decryption(
    final_x,
    final_z,
    n_qubits: int,
):
    # (X^a Z^b)^dagger = Z^b X^a; circuit order is X then Z.
    for wire in range(n_qubits):
        qml.RX(math.pi * final_x[wire], wires=wire)
        qml.RZ(math.pi * final_z[wire], wires=wire)


_qotph_analytic_device = qml.device(
    "default.qubit",
    wires=N_QUBITS,
)


@qml.qnode(
    _qotph_analytic_device,
    interface="torch",
    diff_method="backprop",
)
def qotph_algorithm1_analytic_circuit(
    inputs,
    weights,
    initial_x,
    initial_z,
    layer_signs,
    final_x,
    final_z,
):
    _apply_qotp_encryption(
        initial_x,
        initial_z,
        N_QUBITS,
    )

    for layer_idx in range(RU_LAYERS):
        for wire in range(N_QUBITS):
            angle_index = layer_idx * N_QUBITS + wire
            sign = layer_signs[layer_idx, wire]
            qml.RY(
                sign * inputs[..., angle_index],
                wires=wire,
            )

        for wire in range(N_QUBITS):
            sign = layer_signs[layer_idx, wire]
            qml.RY(
                sign * weights[layer_idx, wire],
                wires=wire,
            )

        for control in range(N_QUBITS):
            target = (control + 1) % N_QUBITS
            qml.CNOT(wires=[control, target])

    _apply_qotp_decryption(
        final_x,
        final_z,
        N_QUBITS,
    )

    return [
        qml.expval(qml.PauliZ(wire))
        for wire in range(N_QUBITS)
    ]


@qml.qnode(
    _qotph_analytic_device,
    interface="torch",
    diff_method="backprop",
)
def qotph_algorithm2_encrypted_analytic_circuit(
    inputs,
    weights,
    initial_x,
    initial_z,
    layer_signs,
):
    _apply_qotp_encryption(
        initial_x,
        initial_z,
        N_QUBITS,
    )

    for layer_idx in range(RU_LAYERS):
        for wire in range(N_QUBITS):
            angle_index = layer_idx * N_QUBITS + wire
            sign = layer_signs[layer_idx, wire]
            qml.RY(
                sign * inputs[..., angle_index],
                wires=wire,
            )

        for wire in range(N_QUBITS):
            sign = layer_signs[layer_idx, wire]
            qml.RY(
                sign * weights[layer_idx, wire],
                wires=wire,
            )

        for control in range(N_QUBITS):
            target = (control + 1) % N_QUBITS
            qml.CNOT(wires=[control, target])

    # The backend returns encrypted observables. The client applies the
    # final X-key correction outside the delegated circuit.
    return [
        qml.expval(qml.PauliZ(wire))
        for wire in range(N_QUBITS)
    ]


_qotph_counts_device = qml.device(
    "default.qubit",
    wires=N_QUBITS,
    shots=QOTPH_SHOTS,
)


@qml.qnode(_qotph_counts_device)
def qotph_algorithm1_counts_circuit(
    inputs,
    weights,
    initial_x,
    initial_z,
    layer_signs,
    final_x,
    final_z,
):
    _apply_qotp_encryption(
        initial_x,
        initial_z,
        N_QUBITS,
    )

    for layer_idx in range(RU_LAYERS):
        for wire in range(N_QUBITS):
            angle_index = layer_idx * N_QUBITS + wire
            qml.RY(
                layer_signs[layer_idx, wire]
                * inputs[angle_index],
                wires=wire,
            )

        for wire in range(N_QUBITS):
            qml.RY(
                layer_signs[layer_idx, wire]
                * weights[layer_idx, wire],
                wires=wire,
            )

        for control in range(N_QUBITS):
            target = (control + 1) % N_QUBITS
            qml.CNOT(wires=[control, target])

    _apply_qotp_decryption(
        final_x,
        final_z,
        N_QUBITS,
    )
    return qml.counts(wires=range(N_QUBITS))


@qml.qnode(_qotph_counts_device)
def qotph_algorithm2_counts_circuit(
    inputs,
    weights,
    initial_x,
    initial_z,
    layer_signs,
):
    _apply_qotp_encryption(
        initial_x,
        initial_z,
        N_QUBITS,
    )

    for layer_idx in range(RU_LAYERS):
        for wire in range(N_QUBITS):
            angle_index = layer_idx * N_QUBITS + wire
            qml.RY(
                layer_signs[layer_idx, wire]
                * inputs[angle_index],
                wires=wire,
            )

        for wire in range(N_QUBITS):
            qml.RY(
                layer_signs[layer_idx, wire]
                * weights[layer_idx, wire],
                wires=wire,
            )

        for control in range(N_QUBITS):
            target = (control + 1) % N_QUBITS
            qml.CNOT(wires=[control, target])

    return qml.counts(wires=range(N_QUBITS))


def normalize_count_word(word) -> str:
    if isinstance(word, str):
        return word.replace(" ", "")
    if isinstance(word, tuple):
        return "".join(str(int(bit)) for bit in word)
    return str(word).replace(" ", "")


def decrypt_algorithm2_counts(
    encrypted_counts: Dict,
    final_keys: QOTPKeys,
    wire_order: Tuple[int, ...],
) -> Dict[str, int]:
    """
    In computational-basis measurement, the final X-key flips the measured
    bit. The Z-key changes phase but not the measurement probability.
    """
    decrypted: Dict[str, int] = {}

    for raw_word, raw_count in encrypted_counts.items():
        word = normalize_count_word(raw_word)
        bits = [int(bit) for bit in word]

        if len(bits) != len(wire_order):
            raise ValueError(
                f"Expected {len(wire_order)} measured bits, got {word!r}"
            )

        for wire in range(len(wire_order)):
            position = wire_order[wire]
            bits[position] ^= final_keys.x[wire]

        plain_word = "".join(str(bit) for bit in bits)
        decrypted[plain_word] = (
            decrypted.get(plain_word, 0)
            + int(raw_count)
        )

    return decrypted


def counts_to_pauli_z(
    counts: Dict,
    n_qubits: int,
    wire_order: Tuple[int, ...],
) -> torch.Tensor:
    total = max(sum(int(value) for value in counts.values()), 1)
    expectations = torch.zeros(
        n_qubits,
        dtype=torch.float32,
    )

    for raw_word, raw_count in counts.items():
        word = normalize_count_word(raw_word)
        probability = float(raw_count) / float(total)

        for wire in range(n_qubits):
            bit = int(word[wire_order[wire]])
            expectations[wire] += probability * (
                1.0 if bit == 0 else -1.0
            )

    return expectations


def _qotph_tensor_schedule(
    initial_keys: QOTPKeys,
    dtype: torch.dtype,
):
    layer_signs, final_keys = qotph_key_schedule(
        initial_keys,
        n_qubits=N_QUBITS,
        reupload_layers=RU_LAYERS,
    )

    initial_x = torch.tensor(
        initial_keys.x,
        dtype=dtype,
    )
    initial_z = torch.tensor(
        initial_keys.z,
        dtype=dtype,
    )
    signs = torch.tensor(
        layer_signs,
        dtype=dtype,
    )
    final_x = torch.tensor(
        final_keys.x,
        dtype=dtype,
    )
    final_z = torch.tensor(
        final_keys.z,
        dtype=dtype,
    )

    return (
        initial_x,
        initial_z,
        signs,
        final_x,
        final_z,
        final_keys,
    )


QOTPH_RUNTIME_STATS = defaultdict(float)


def reset_qotph_runtime_stats():
    QOTPH_RUNTIME_STATS.clear()


def snapshot_qotph_runtime_stats() -> Dict[str, float]:
    return {
        key: float(value)
        for key, value in QOTPH_RUNTIME_STATS.items()
    }


def qotph_analytic_features(
    inputs: torch.Tensor,
    weights: torch.Tensor,
    initial_keys: QOTPKeys,
    mode: str,
) -> torch.Tensor:
    (
        initial_x,
        initial_z,
        signs,
        final_x,
        final_z,
        _,
    ) = _qotph_tensor_schedule(
        initial_keys,
        dtype=inputs.dtype,
    )

    started = time.perf_counter()

    if mode == "algorithm1":
        output = qotph_algorithm1_analytic_circuit(
            inputs,
            weights,
            initial_x,
            initial_z,
            signs,
            final_x,
            final_z,
        )
    elif mode == "algorithm2":
        encrypted_output = (
            qotph_algorithm2_encrypted_analytic_circuit(
                inputs,
                weights,
                initial_x,
                initial_z,
                signs,
            )
        )
        output = (
            torch.stack(list(encrypted_output), dim=-1)
            if isinstance(encrypted_output, (tuple, list))
            else encrypted_output
        )

        correction = 1.0 - 2.0 * final_x
        output = output * correction
    else:
        raise ValueError("QOTPH analytic mode must be algorithm1 or algorithm2")

    if isinstance(output, (tuple, list)):
        output = torch.stack(list(output), dim=-1)

    QOTPH_RUNTIME_STATS["analytic_protected_forward_seconds"] += (
        time.perf_counter() - started
    )
    QOTPH_RUNTIME_STATS["analytic_protected_forward_calls"] += 1
    return output


def qotph_counts_features_single(
    inputs: torch.Tensor,
    weights: torch.Tensor,
    initial_keys: QOTPKeys,
    mode: str,
) -> torch.Tensor:
    (
        initial_x,
        initial_z,
        signs,
        final_x,
        final_z,
        final_keys,
    ) = _qotph_tensor_schedule(
        initial_keys,
        dtype=torch.float64,
    )

    inputs_np = inputs.detach().cpu().double().numpy()
    weights_np = weights.detach().cpu().double().numpy()
    initial_x_np = initial_x.numpy()
    initial_z_np = initial_z.numpy()
    signs_np = signs.numpy()
    final_x_np = final_x.numpy()
    final_z_np = final_z.numpy()

    started = time.perf_counter()
    if mode == "algorithm1":
        counts = qotph_algorithm1_counts_circuit(
            inputs_np,
            weights_np,
            initial_x_np,
            initial_z_np,
            signs_np,
            final_x_np,
            final_z_np,
        )
        local_started = time.perf_counter()
        plain_counts = {
            normalize_count_word(word): int(count)
            for word, count in counts.items()
        }
    elif mode == "algorithm2":
        counts = qotph_algorithm2_counts_circuit(
            inputs_np,
            weights_np,
            initial_x_np,
            initial_z_np,
            signs_np,
        )
        local_started = time.perf_counter()
        plain_counts = decrypt_algorithm2_counts(
            counts,
            final_keys,
            QOTPH_WIRE_ORDER,
        )
    else:
        raise ValueError("QOTPH counts mode must be algorithm1 or algorithm2")

    QOTPH_RUNTIME_STATS["counts_backend_seconds"] += (
        local_started - started
    )

    features = counts_to_pauli_z(
        plain_counts,
        n_qubits=N_QUBITS,
        wire_order=QOTPH_WIRE_ORDER,
    )

    QOTPH_RUNTIME_STATS["local_decryption_seconds"] += (
        time.perf_counter() - local_started
    )
    QOTPH_RUNTIME_STATS["counts_circuit_calls"] += 1
    QOTPH_RUNTIME_STATS["shots_executed"] += QOTPH_SHOTS
    return features


def qotph_counts_features_batch(
    inputs: torch.Tensor,
    weights: torch.Tensor,
    initial_keys: QOTPKeys,
    mode: str,
) -> torch.Tensor:
    rows = [
        qotph_counts_features_single(
            inputs[row],
            weights,
            initial_keys,
            mode,
        )
        for row in range(inputs.shape[0])
    ]
    return torch.stack(rows, dim=0).to(dtype=inputs.dtype)


class QOTPHCountsParameterShift(torch.autograd.Function):
    """
    Paper-aligned finite-shot Algorithm-1/2 training path.

    The backend returns sampled counts; the client locally reconstructs
    Pauli-Z features. Fresh gradients are estimated using parameter shift
    for both adapter-generated angles and trainable quantum weights.

    This is intentionally extremely expensive and should first be used with
    RUN_PROFILE="qotph_counts_smoke".
    """

    @staticmethod
    def forward(
        ctx,
        inputs: torch.Tensor,
        weights: torch.Tensor,
        initial_x: torch.Tensor,
        initial_z: torch.Tensor,
        mode_code: torch.Tensor,
    ):
        initial_keys = QOTPKeys(
            tuple(int(v) for v in initial_x.cpu().tolist()),
            tuple(int(v) for v in initial_z.cpu().tolist()),
        )
        mode = (
            "algorithm1"
            if int(mode_code.item()) == 1
            else "algorithm2"
        )

        output = qotph_counts_features_batch(
            inputs,
            weights,
            initial_keys,
            mode,
        )

        ctx.mode = mode
        ctx.save_for_backward(
            inputs.detach(),
            weights.detach(),
            initial_x.detach(),
            initial_z.detach(),
        )
        return output

    @staticmethod
    def backward(ctx, grad_output):
        (
            inputs,
            weights,
            initial_x,
            initial_z,
        ) = ctx.saved_tensors

        initial_keys = QOTPKeys(
            tuple(int(v) for v in initial_x.cpu().tolist()),
            tuple(int(v) for v in initial_z.cpu().tolist()),
        )

        shift = math.pi / 2.0
        grad_inputs = torch.zeros_like(inputs)
        grad_weights = torch.zeros_like(weights)

        started = time.perf_counter()

        for row in range(inputs.shape[0]):
            for angle_index in range(inputs.shape[1]):
                plus_inputs = inputs[row].clone()
                minus_inputs = inputs[row].clone()
                plus_inputs[angle_index] += shift
                minus_inputs[angle_index] -= shift

                plus_output = qotph_counts_features_single(
                    plus_inputs,
                    weights,
                    initial_keys,
                    ctx.mode,
                )
                minus_output = qotph_counts_features_single(
                    minus_inputs,
                    weights,
                    initial_keys,
                    ctx.mode,
                )

                derivative = 0.5 * (
                    plus_output - minus_output
                ).to(dtype=grad_output.dtype)

                grad_inputs[row, angle_index] = torch.dot(
                    grad_output[row].cpu(),
                    derivative.cpu(),
                ).to(dtype=inputs.dtype)

        for layer_idx in range(weights.shape[0]):
            for wire in range(weights.shape[1]):
                plus_weights = weights.clone()
                minus_weights = weights.clone()
                plus_weights[layer_idx, wire] += shift
                minus_weights[layer_idx, wire] -= shift

                accumulated = weights.new_zeros(())
                for row in range(inputs.shape[0]):
                    plus_output = qotph_counts_features_single(
                        inputs[row],
                        plus_weights,
                        initial_keys,
                        ctx.mode,
                    )
                    minus_output = qotph_counts_features_single(
                        inputs[row],
                        minus_weights,
                        initial_keys,
                        ctx.mode,
                    )

                    derivative = 0.5 * (
                        plus_output - minus_output
                    ).to(dtype=grad_output.dtype)

                    accumulated = accumulated + torch.dot(
                        grad_output[row].cpu(),
                        derivative.cpu(),
                    ).to(dtype=weights.dtype)

                grad_weights[layer_idx, wire] = accumulated

        QOTPH_RUNTIME_STATS["parameter_shift_backward_seconds"] += (
            time.perf_counter() - started
        )
        QOTPH_RUNTIME_STATS["parameter_shift_backward_calls"] += 1

        return (
            grad_inputs,
            grad_weights,
            None,
            None,
            None,
        )


def qotph_counts_parameter_shift_features(
    inputs: torch.Tensor,
    weights: torch.Tensor,
    initial_keys: QOTPKeys,
    mode: str,
) -> torch.Tensor:
    if (
        inputs.shape[0] > QOTPH_COUNTS_MAX_BATCH
        and not ALLOW_EXPENSIVE_COUNTS_TRAINING
    ):
        raise RuntimeError(
            "Finite-shot QOTPH parameter-shift training is guarded. "
            f"Current batch={inputs.shape[0]}, allowed="
            f"{QOTPH_COUNTS_MAX_BATCH}. Use the counts smoke profile or "
            "set ALLOW_EXPENSIVE_COUNTS_TRAINING=True explicitly."
        )

    mode_code = torch.tensor(
        1 if mode == "algorithm1" else 2,
        dtype=torch.int64,
        device=inputs.device,
    )
    initial_x = torch.tensor(
        initial_keys.x,
        dtype=torch.int64,
        device=inputs.device,
    )
    initial_z = torch.tensor(
        initial_keys.z,
        dtype=torch.int64,
        device=inputs.device,
    )

    return QOTPHCountsParameterShift.apply(
        inputs,
        weights,
        initial_x,
        initial_z,
        mode_code,
    )


def hellinger_distance(
    p: Dict[str, float],
    q: Dict[str, float],
) -> float:
    keys = sorted(set(p) | set(q))
    return float(
        math.sqrt(
            0.5
            * sum(
                (
                    math.sqrt(max(p.get(key, 0.0), 0.0))
                    - math.sqrt(max(q.get(key, 0.0), 0.0))
                )
                ** 2
                for key in keys
            )
        )
    )


def total_variation_distance(
    p: Dict[str, float],
    q: Dict[str, float],
) -> float:
    keys = sorted(set(p) | set(q))
    return float(
        0.5
        * sum(
            abs(p.get(key, 0.0) - q.get(key, 0.0))
            for key in keys
        )
    )


# Hybrid model

The classical encoder produces a 16-dimensional latent vector. Four independent adapters feed four parallel 4-qubit, 2-layer data-reuploading circuits. Two trainable quantum tensors are shared cyclically across the four circuits. The 16 recovered quantum features are concatenated with the 16 classical skip features before fusion and classification.

Resource-constrained clients preserve the same global tensor shapes by evaluating only selected circuits and zero-filling inactive quantum feature groups.


In [19]:

# ============================================================
# Original AdeptHEQ baseline and proposed protected QBank model
# ============================================================

_original_dev = qml.device(
    "default.qubit",
    wires=N_QUBITS,
)
_original_weight_shapes = {
    "weights": (2, N_QUBITS, 3)
}


@qml.qnode(
    _original_dev,
    interface="torch",
    diff_method="backprop",
)
def original_quantum_net(inputs, weights):
    qml.AmplitudeEmbedding(
        features=inputs,
        wires=range(N_QUBITS),
        pad_with=0.0,
        normalize=True,
    )
    qml.StronglyEntanglingLayers(
        weights,
        wires=range(N_QUBITS),
    )
    return [
        qml.expval(qml.PauliZ(wire))
        for wire in range(N_QUBITS)
    ]


class OriginalAdeptNet(nn.Module):
    """Original 4-qubit, 2-layer Fashion-MNIST AdeptHEQ model."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 2**N_QUBITS),
        )
        self.qnn = qml.qnn.TorchLayer(
            original_quantum_net,
            _original_weight_shapes,
        )
        self.fc4 = nn.Linear(
            N_QUBITS,
            num_classes,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.network(x)
        x = self.qnn(x)
        return self.fc4(x)


# ============================================================
# Plain parallel data-reuploading circuit
# ============================================================

_reupload_dev = qml.device(
    "default.qubit",
    wires=N_QUBITS,
)


@qml.qnode(
    _reupload_dev,
    interface="torch",
    diff_method="backprop",
)
def data_reuploading_circuit(inputs, weights):
    for layer_idx in range(RU_LAYERS):
        start = layer_idx * N_QUBITS
        stop = (layer_idx + 1) * N_QUBITS

        qml.AngleEmbedding(
            inputs[..., start:stop],
            wires=range(N_QUBITS),
            rotation="Y",
        )

        for wire in range(N_QUBITS):
            qml.RY(
                weights[layer_idx, wire],
                wires=wire,
            )

        for control in range(N_QUBITS):
            target = (control + 1) % N_QUBITS
            qml.CNOT(wires=[control, target])

    return [
        qml.expval(qml.PauliZ(wire))
        for wire in range(N_QUBITS)
    ]


class AdeptCNNEncoder(nn.Module):
    """
    Original AdeptHEQ convolutional backbone with a compact 16-D latent.
    """

    def __init__(
        self,
        latent_dim: int = LATENT_DIM,
    ):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.Conv2d(
                256,
                256,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(256 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim),
            nn.Tanh(),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.network(x)


class ParallelDataReuploadingQBank(nn.Module):
    """
    Four circuit-specific adapters and four parallel 4-qubit circuits.

    Two unique quantum-weight tensors are shared cyclically:
        circuits 0/2 -> tensor 0
        circuits 1/3 -> tensor 1

    QOTPH wraps each active circuit without changing the 16-D quantum
    feature interface. Resource-constrained clients can evaluate a rotating
    subset of circuits while inactive feature groups are zero-filled.
    """

    def __init__(
        self,
        latent_dim: int = LATENT_DIM,
        k_circuits: int = K_CIRCUITS,
        r_unique: int = R_UNIQUE,
    ):
        super().__init__()

        if r_unique > k_circuits:
            raise ValueError(
                "R_UNIQUE cannot exceed K_CIRCUITS"
            )

        self.k_circuits = int(k_circuits)
        self.r_unique = int(r_unique)
        features_per_circuit = (
            RU_LAYERS * N_QUBITS
        )

        self.adapters = nn.ModuleList(
            [
                nn.Linear(
                    latent_dim,
                    features_per_circuit,
                )
                for _ in range(k_circuits)
            ]
        )

        self.q_weights = nn.ParameterList(
            [
                nn.Parameter(
                    0.01
                    * torch.randn(
                        RU_LAYERS,
                        N_QUBITS,
                    )
                )
                for _ in range(r_unique)
            ]
        )

        self.active_circuit_ids = tuple(
            range(self.k_circuits)
        )
        self.client_id = 0
        self.round_number = 0
        self._qotph_invocation_counter = 0

    def _shared_weight_id(
        self,
        circuit_id: int,
    ) -> int:
        return circuit_id % self.r_unique

    def set_execution_context(
        self,
        client_id: int,
        round_number: int,
        active_circuit_ids: Optional[Tuple[int, ...]] = None,
    ) -> None:
        self.client_id = int(client_id)
        self.round_number = int(round_number)
        self._qotph_invocation_counter = 0

        if active_circuit_ids is None:
            self.active_circuit_ids = tuple(
                range(self.k_circuits)
            )
        else:
            active = tuple(
                sorted(
                    set(
                        int(cid)
                        for cid in active_circuit_ids
                    )
                )
            )
            if not active:
                raise ValueError(
                    "At least one circuit must remain active"
                )
            if min(active) < 0 or max(active) >= self.k_circuits:
                raise ValueError(
                    "active_circuit_ids contains an invalid circuit"
                )
            self.active_circuit_ids = active

    @staticmethod
    def _stack_qnode_output(
        output,
    ) -> torch.Tensor:
        if isinstance(output, (tuple, list)):
            return torch.stack(
                list(output),
                dim=-1,
            )
        return output

    def _keys_for(
        self,
        circuit_id: int,
        invocation_id: int,
        sample_id: int = 0,
    ) -> QOTPKeys:
        if QOTPH_KEY_MODE == "secure_random":
            return QOTPKeys.secure_random(N_QUBITS)

        return deterministic_qotp_keys(
            key_seed=QOTPH_KEY_SEED,
            client_id=self.client_id,
            round_number=self.round_number,
            invocation_id=(
                int(invocation_id) * 1_000_000
                + int(sample_id)
            ),
            circuit_id=circuit_id,
            n_qubits=N_QUBITS,
        )

    def _run_one_circuit(
        self,
        features: torch.Tensor,
        weights: torch.Tensor,
        detach_circuit: bool,
        circuit_id: int,
        invocation_id: int,
    ) -> torch.Tensor:
        original_device = features.device
        original_dtype = features.dtype

        features_cpu = features.to("cpu")
        weights_cpu = weights.to("cpu")

        if QOTPH_MODE == "off":
            if detach_circuit:
                with torch.no_grad():
                    output = data_reuploading_circuit(
                        features_cpu.detach(),
                        weights_cpu.detach(),
                    )
            else:
                output = data_reuploading_circuit(
                    features_cpu,
                    weights_cpu,
                )

            output = self._stack_qnode_output(output)

        else:
            # A quantum one-time pad must not be reused across independent
            # encrypted inputs. Each sample therefore receives its own key
            # pair for this circuit invocation. This is slower than one
            # broadcast QNode call, but it preserves the intended privacy
            # semantics and keeps the full batch gradient connected.
            protected_rows = []

            for sample_id in range(features_cpu.shape[0]):
                keys = self._keys_for(
                    circuit_id=circuit_id,
                    invocation_id=invocation_id,
                    sample_id=sample_id,
                )
                sample_features = features_cpu[
                    sample_id:sample_id + 1
                ]

                if QOTPH_EXECUTION == "analytic":
                    if detach_circuit:
                        with torch.no_grad():
                            sample_output = qotph_analytic_features(
                                sample_features[0].detach(),
                                weights_cpu.detach(),
                                keys,
                                QOTPH_MODE,
                            )
                    else:
                        sample_output = qotph_analytic_features(
                            sample_features[0],
                            weights_cpu,
                            keys,
                            QOTPH_MODE,
                        )

                    sample_output = self._stack_qnode_output(
                        sample_output
                    ).reshape(1, N_QUBITS)

                elif (
                    QOTPH_EXECUTION
                    == "finite_shot_parameter_shift"
                ):
                    if detach_circuit:
                        with torch.no_grad():
                            sample_output = (
                                qotph_counts_features_batch(
                                    sample_features.detach(),
                                    weights_cpu.detach(),
                                    keys,
                                    QOTPH_MODE,
                                )
                            )
                    else:
                        sample_output = (
                            qotph_counts_parameter_shift_features(
                                sample_features,
                                weights_cpu,
                                keys,
                                QOTPH_MODE,
                            )
                        )
                else:
                    raise ValueError(
                        f"Unsupported QOTPH_EXECUTION="
                        f"{QOTPH_EXECUTION!r}"
                    )

                protected_rows.append(sample_output)

            output = torch.cat(protected_rows, dim=0)

        return output.to(
            device=original_device,
            dtype=original_dtype,
        )

    def forward(
        self,
        latent: torch.Tensor,
        detach_circuit: bool = False,
        active_circuit_ids: Optional[Tuple[int, ...]] = None,
    ) -> torch.Tensor:
        if active_circuit_ids is None:
            active_circuit_ids = self.active_circuit_ids

        active_set = set(active_circuit_ids)
        invocation_id = self._qotph_invocation_counter
        self._qotph_invocation_counter += 1

        quantum_features = []

        for circuit_id, adapter in enumerate(
            self.adapters
        ):
            if circuit_id not in active_set:
                quantum_features.append(
                    latent.new_zeros(
                        latent.shape[0],
                        N_QUBITS,
                    )
                )
                continue

            angles = (
                torch.tanh(adapter(latent))
                * math.pi
            )
            shared_weights = self.q_weights[
                self._shared_weight_id(
                    circuit_id
                )
            ]

            q_out = self._run_one_circuit(
                angles,
                shared_weights,
                detach_circuit=detach_circuit,
                circuit_id=circuit_id,
                invocation_id=invocation_id,
            )
            quantum_features.append(q_out)

        return torch.cat(
            quantum_features,
            dim=1,
        )


class ParallelQBankAdeptNet(nn.Module):
    def __init__(
        self,
        num_classes: int = 10,
    ):
        super().__init__()
        self.encoder = AdeptCNNEncoder(
            latent_dim=LATENT_DIM
        )
        self.qbank = ParallelDataReuploadingQBank(
            latent_dim=LATENT_DIM
        )

        fused_dim = (
            LATENT_DIM
            + K_CIRCUITS * N_QUBITS
        )
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.classifier = nn.Linear(
            64,
            num_classes,
        )

    def set_execution_context(
        self,
        client_id: int,
        round_number: int,
        active_circuit_ids: Optional[Tuple[int, ...]] = None,
    ) -> None:
        self.qbank.set_execution_context(
            client_id=client_id,
            round_number=round_number,
            active_circuit_ids=active_circuit_ids,
        )

    def encode(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        return self.encoder(x)

    def quantum_features(
        self,
        latent: torch.Tensor,
        detach_circuit: bool = False,
        active_circuit_ids: Optional[Tuple[int, ...]] = None,
    ) -> torch.Tensor:
        return self.qbank(
            latent,
            detach_circuit=detach_circuit,
            active_circuit_ids=active_circuit_ids,
        )

    def classify(
        self,
        latent: torch.Tensor,
        quantum_features: torch.Tensor,
    ) -> torch.Tensor:
        quantum_features = quantum_features.to(
            device=latent.device,
            dtype=latent.dtype,
        )
        fused = torch.cat(
            [
                latent,
                quantum_features,
            ],
            dim=1,
        )
        return self.classifier(
            self.fusion(fused)
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        latent = self.encode(x)
        q_features = self.quantum_features(
            latent,
            detach_circuit=(
                not torch.is_grad_enabled()
            ),
        )
        return self.classify(
            latent,
            q_features,
        )


def build_model(
    num_classes: int,
):
    if MODEL_VARIANT == "original":
        if QUANTUM_TRAINING != "naive":
            raise ValueError(
                "The original baseline currently uses standard "
                "full backpropagation. Set QUANTUM_TRAINING='naive'."
            )
        if QOTPH_MODE != "off":
            raise ValueError(
                "QOTPH is integrated into the parallel QBank model. "
                "Use MODEL_VARIANT='parallel_qbank' for Stage 2."
            )
        return OriginalAdeptNet(
            num_classes=num_classes
        )

    if MODEL_VARIANT == "parallel_qbank":
        return ParallelQBankAdeptNet(
            num_classes=num_classes
        )

    raise ValueError(
        "MODEL_VARIANT must be 'original' "
        "or 'parallel_qbank'"
    )


In [9]:

# ============================================================
# QOTPH and quantum-classical interface tests
# ============================================================

def run_qotph_analytic_equivalence_test(
    trials: int = 3,
) -> Dict[str, float]:
    rng = np.random.default_rng(seed + 9001)
    max_alg1_error = 0.0
    max_alg2_error = 0.0

    for trial in range(trials):
        angles = torch.tensor(
            rng.normal(
                size=(1, RU_LAYERS * N_QUBITS)
            ),
            dtype=torch.float64,
        )
        weights = torch.tensor(
            rng.normal(
                size=(RU_LAYERS, N_QUBITS)
            ),
            dtype=torch.float64,
        )

        plain = data_reuploading_circuit(
            angles,
            weights,
        )
        plain = (
            torch.stack(list(plain), dim=-1)
            if isinstance(plain, (tuple, list))
            else plain
        )

        keys = deterministic_qotp_keys(
            QOTPH_KEY_SEED,
            client_id=0,
            round_number=0,
            invocation_id=trial,
            circuit_id=0,
            n_qubits=N_QUBITS,
        )

        alg1 = qotph_analytic_features(
            angles,
            weights,
            keys,
            "algorithm1",
        )
        alg2 = qotph_analytic_features(
            angles,
            weights,
            keys,
            "algorithm2",
        )

        max_alg1_error = max(
            max_alg1_error,
            float(
                torch.max(
                    torch.abs(plain - alg1)
                ).item()
            ),
        )
        max_alg2_error = max(
            max_alg2_error,
            float(
                torch.max(
                    torch.abs(plain - alg2)
                ).item()
            ),
        )

    if max_alg1_error > 1e-5:
        raise AssertionError(
            f"Algorithm-1 analytic equivalence failed: "
            f"{max_alg1_error:.3e}"
        )

    if max_alg2_error > 1e-5:
        raise AssertionError(
            f"Algorithm-2 analytic local correction failed: "
            f"{max_alg2_error:.3e}"
        )

    return {
        "algorithm1_max_abs_error": max_alg1_error,
        "algorithm2_max_abs_error": max_alg2_error,
    }


def run_qotph_count_decryption_sanity() -> None:
    # Synthetic wire-order sanity: X-key on wire 0 maps 1000 -> 0000
    # when QOTPH_WIRE_ORDER is the identity ordering.
    test_keys = QOTPKeys(
        x=(1, 0, 0, 0),
        z=(0, 0, 0, 0),
    )
    decrypted = decrypt_algorithm2_counts(
        {"1000": 10},
        test_keys,
        QOTPH_WIRE_ORDER,
    )

    if decrypted != {"0000": 10}:
        raise AssertionError(
            "Configured QOTPH_WIRE_ORDER failed the synthetic "
            f"decryption check: {decrypted}"
        )


def run_qotph_finite_shot_feature_test() -> Dict[str, float]:
    rng = np.random.default_rng(seed + 9017)
    angles = torch.tensor(
        rng.normal(
            size=(1, RU_LAYERS * N_QUBITS)
        ),
        dtype=torch.float64,
    )
    weights = torch.tensor(
        rng.normal(
            size=(RU_LAYERS, N_QUBITS)
        ),
        dtype=torch.float64,
    )
    keys = deterministic_qotp_keys(
        QOTPH_KEY_SEED,
        client_id=0,
        round_number=0,
        invocation_id=99,
        circuit_id=0,
        n_qubits=N_QUBITS,
    )

    plain = data_reuploading_circuit(
        angles,
        weights,
    )
    plain = (
        torch.stack(list(plain), dim=-1)
        if isinstance(plain, (tuple, list))
        else plain
    ).squeeze(0).float()

    alg1 = qotph_counts_features_single(
        angles.squeeze(0),
        weights,
        keys,
        "algorithm1",
    )
    alg2 = qotph_counts_features_single(
        angles.squeeze(0),
        weights,
        keys,
        "algorithm2",
    )

    return {
        "algorithm1_expectation_mae": float(
            torch.mean(torch.abs(plain - alg1)).item()
        ),
        "algorithm2_expectation_mae": float(
            torch.mean(torch.abs(plain - alg2)).item()
        ),
        "algorithm1_vs_algorithm2_mae": float(
            torch.mean(torch.abs(alg1 - alg2)).item()
        ),
    }


qotph_unit_test_results = {}

if RUN_QOTPH_UNIT_TESTS:
    reset_qotph_runtime_stats()
    run_qotph_count_decryption_sanity()
    qotph_unit_test_results.update(
        run_qotph_analytic_equivalence_test()
    )

    # Finite-shot checks are intentionally run only for the dedicated
    # counts profile, avoiding an unnecessary sampling delay otherwise.
    if RUN_PROFILE == "qotph_counts_smoke":
        qotph_unit_test_results.update(
            run_qotph_finite_shot_feature_test()
        )

    print("QOTPH unit-test results:")
    for key, value in qotph_unit_test_results.items():
        print(f"  {key}: {value:.6e}")


# Interface sanity check independent of client data loaders.
_dtype_test_model = build_model(
    num_classes=len(CLASSES)
).to(DEVICE)

if isinstance(
    _dtype_test_model,
    ParallelQBankAdeptNet,
):
    _dtype_test_model.set_execution_context(
        client_id=0,
        round_number=0,
        active_circuit_ids=tuple(
            range(K_CIRCUITS)
        ),
    )

_dtype_test_batch = torch.zeros(
    2,
    1,
    28,
    28,
    device=DEVICE,
    dtype=torch.float32,
)

_dtype_test_model.eval()

with torch.no_grad():
    _logits = _dtype_test_model(
        _dtype_test_batch
    )

assert _logits.shape == (
    2,
    len(CLASSES),
)
assert torch.isfinite(_logits).all()
assert _logits.dtype == torch.float32

print(
    "Quantum-classical interface passed:",
    _logits.shape,
    _logits.dtype,
    _logits.device,
)

del _dtype_test_model
del _dtype_test_batch
del _logits

if torch.cuda.is_available():
    torch.cuda.empty_cache()


QOTPH unit-test results:
  algorithm1_max_abs_error: 0.000000e+00
  algorithm2_max_abs_error: 2.220446e-16
Quantum-classical interface passed: torch.Size([2, 10]) torch.float32 cuda:0


# Adaptive quantum backward and heterogeneous clients

The adaptive refresh policy combines learning need and client capacity:

\[
N_i=w_e(1-\bar q_i)+w_sS_i+w_dD_i+w_uU_i+w_fF_i
\]

\[
C_i=v_cC_i^{compute}+v_mC_i^{memory}+v_bC_i^{battery}+v_qC_i^{QPU}
\]

\[
r_i=r_{min}+(r_{max}-r_{min})N_i(0.35+0.65C_i)
\]

Safety rules force a fresh protected backward during warm-up, excessive drift, maximum staleness, or minimum-budget violation. Bandwidth affects upload sparsity, while compute, memory, battery, and QPU responsiveness affect local epochs, active circuits, and maximum refresh ratio.


In [20]:

# ============================================================
# Adaptive LazyQ and heterogeneous resource policies
# ============================================================

@dataclass
class ClientResourceProfile:
    tier: str
    compute_score: float
    memory_score: float
    battery_score: float
    qpu_latency_score: float
    bandwidth_score: float


@dataclass
class AdaptiveClientState:
    quality_ema: float = 0.50
    improvement_ema: float = 0.0
    uncertainty_ema: float = 0.50
    fairness_debt: float = 0.0
    last_quality: Optional[float] = None


@dataclass
class AdaptiveRefreshConfig:
    minimum_refresh_ratio: float = MIN_REFRESH_RATIO
    maximum_refresh_ratio: float = MAX_REFRESH_RATIO
    maximum_stale_steps: int = MAX_STALE_STEPS
    warmup_fresh_steps: int = WARMUP_FRESH_STEPS
    drift_threshold: float = TAU_DRIFT
    error_weight: float = ADAPT_ERROR_WEIGHT
    stagnation_weight: float = ADAPT_STAGNATION_WEIGHT
    drift_weight: float = ADAPT_DRIFT_WEIGHT
    uncertainty_weight: float = ADAPT_UNCERTAINTY_WEIGHT
    fairness_weight: float = ADAPT_FAIRNESS_WEIGHT


@dataclass
class ClientExecutionPolicy:
    local_epochs: int
    active_circuit_ids: Tuple[int, ...]
    maximum_refresh_ratio: float
    upload_fraction: float
    capacity_score: float


def clamp01(value: float) -> float:
    return float(
        max(
            0.0,
            min(1.0, value),
        )
    )


def build_client_resource_profiles(
    num_clients: int,
    seed_value: int,
) -> Dict[int, ClientResourceProfile]:
    """
    Deterministic simulated heterogeneous resources.

    The profiles are experimental controls, not claims about physical
    hardware. Calibrate them later with measured client/QPU traces.
    """
    rng = np.random.default_rng(seed_value)
    tier_cycle = [
        "strong",
        "medium",
        "weak",
    ]
    profiles = {}

    ranges = {
        "strong": {
            "compute": (0.80, 1.00),
            "memory": (0.75, 1.00),
            "battery": (0.70, 1.00),
            "qpu": (0.75, 1.00),
            "bandwidth": (0.75, 1.00),
        },
        "medium": {
            "compute": (0.45, 0.75),
            "memory": (0.45, 0.75),
            "battery": (0.40, 0.75),
            "qpu": (0.40, 0.75),
            "bandwidth": (0.35, 0.75),
        },
        "weak": {
            "compute": (0.15, 0.40),
            "memory": (0.20, 0.45),
            "battery": (0.15, 0.45),
            "qpu": (0.15, 0.45),
            "bandwidth": (0.10, 0.40),
        },
    }

    for cid in range(num_clients):
        tier = tier_cycle[cid % len(tier_cycle)]
        bounds = ranges[tier]

        profiles[cid] = ClientResourceProfile(
            tier=tier,
            compute_score=float(
                rng.uniform(*bounds["compute"])
            ),
            memory_score=float(
                rng.uniform(*bounds["memory"])
            ),
            battery_score=float(
                rng.uniform(*bounds["battery"])
            ),
            qpu_latency_score=float(
                rng.uniform(*bounds["qpu"])
            ),
            bandwidth_score=float(
                rng.uniform(*bounds["bandwidth"])
            ),
        )

    return profiles


def resource_capacity_score(
    profile: ClientResourceProfile,
) -> float:
    return clamp01(
        0.45 * profile.compute_score
        + 0.20 * profile.memory_score
        + 0.20 * profile.battery_score
        + 0.15 * profile.qpu_latency_score
    )


def allocate_client_policy(
    profile: ClientResourceProfile,
    client_id: int,
    round_number: int,
    base_local_epochs: int,
    k_circuits: int,
) -> ClientExecutionPolicy:
    if not USE_RESOURCE_AWARE_POLICY:
        return ClientExecutionPolicy(
            local_epochs=base_local_epochs,
            active_circuit_ids=tuple(
                range(k_circuits)
            ),
            maximum_refresh_ratio=MAX_REFRESH_RATIO,
            upload_fraction=1.0,
            capacity_score=1.0,
        )

    capacity = resource_capacity_score(
        profile
    )

    if capacity >= 0.75:
        epoch_fraction = 1.00
        active_count = k_circuits
        max_refresh = min(
            MAX_REFRESH_RATIO,
            0.50,
        )
    elif capacity >= 0.40:
        epoch_fraction = 2.0 / 3.0
        active_count = max(
            1,
            int(math.ceil(k_circuits / 2)),
        )
        max_refresh = min(
            MAX_REFRESH_RATIO,
            0.30,
        )
    else:
        epoch_fraction = 1.0 / 3.0
        active_count = 1
        max_refresh = min(
            MAX_REFRESH_RATIO,
            0.15,
        )

    local_epochs = max(
        1,
        int(
            math.ceil(
                base_local_epochs
                * epoch_fraction
            )
        ),
    )

    if USE_CIRCUIT_MASKING:
        start = (
            int(client_id)
            + int(round_number)
        ) % k_circuits
        active_circuits = tuple(
            (start + offset) % k_circuits
            for offset in range(active_count)
        )
    else:
        active_circuits = tuple(
            range(k_circuits)
        )

    if profile.bandwidth_score >= 0.70:
        upload_fraction = 1.0
    elif profile.bandwidth_score >= 0.35:
        upload_fraction = 0.60
    else:
        upload_fraction = 0.35

    return ClientExecutionPolicy(
        local_epochs=local_epochs,
        active_circuit_ids=active_circuits,
        maximum_refresh_ratio=max_refresh,
        upload_fraction=upload_fraction,
        capacity_score=capacity,
    )


def adaptive_refresh_interval(
    state: AdaptiveClientState,
    profile: ClientResourceProfile,
    normalized_drift: float,
    config: AdaptiveRefreshConfig,
    maximum_ratio_override: Optional[float] = None,
):
    error_need = 1.0 - clamp01(
        state.quality_ema
    )
    stagnation_need = clamp01(
        -state.improvement_ema
    )
    drift_need = clamp01(
        normalized_drift
    )
    uncertainty_need = clamp01(
        state.uncertainty_ema
    )
    fairness_need = clamp01(
        state.fairness_debt
    )

    learning_need = (
        config.error_weight * error_need
        + config.stagnation_weight
        * stagnation_need
        + config.drift_weight * drift_need
        + config.uncertainty_weight
        * uncertainty_need
        + config.fairness_weight
        * fairness_need
    )

    capacity = resource_capacity_score(
        profile
    )

    maximum_ratio = (
        config.maximum_refresh_ratio
        if maximum_ratio_override is None
        else min(
            config.maximum_refresh_ratio,
            maximum_ratio_override,
        )
    )

    target_ratio = (
        config.minimum_refresh_ratio
        + (
            maximum_ratio
            - config.minimum_refresh_ratio
        )
        * learning_need
        * (
            0.35
            + 0.65 * capacity
        )
    )

    target_ratio = max(
        config.minimum_refresh_ratio,
        min(
            maximum_ratio,
            target_ratio,
        ),
    )

    interval = int(
        round(
            1.0
            / max(
                target_ratio,
                1e-8,
            )
        )
    )

    interval = max(
        2,
        min(
            interval,
            config.maximum_stale_steps,
        ),
    )

    return (
        interval,
        float(target_ratio),
        float(learning_need),
        float(capacity),
    )


def should_refresh_quantum_backward(
    local_step: int,
    last_refresh_step: int,
    fresh_count: int,
    drift: float,
    interval: int,
    config: AdaptiveRefreshConfig,
):
    age = (
        local_step
        - last_refresh_step
    )

    if local_step < config.warmup_fresh_steps:
        return True, "warmup"

    if drift > config.drift_threshold:
        return True, "drift"

    if age >= config.maximum_stale_steps:
        return True, "maximum_staleness"

    required_fresh = math.ceil(
        config.minimum_refresh_ratio
        * (
            local_step
            + 1
        )
    )

    if fresh_count < required_fresh:
        return True, "minimum_budget"

    if age >= interval:
        return True, "adaptive_interval"

    return False, "cached_gradient"


def normalized_predictive_entropy(
    probabilities: np.ndarray,
) -> float:
    if probabilities.size == 0:
        return 0.0

    clipped = np.clip(
        probabilities,
        1e-12,
        1.0,
    )
    entropy = -np.sum(
        clipped * np.log(clipped),
        axis=1,
    )
    normalizer = math.log(
        probabilities.shape[1]
    )
    return float(
        np.mean(entropy)
        / max(normalizer, 1e-12)
    )


def update_adaptive_client_state(
    state: AdaptiveClientState,
    current_quality: float,
    uncertainty: float,
    actual_refresh_ratio: float,
    target_refresh_ratio: float,
) -> AdaptiveClientState:
    previous_quality = (
        state.last_quality
        if state.last_quality is not None
        else state.quality_ema
    )
    improvement = (
        float(current_quality)
        - float(previous_quality)
    )

    quality_ema = (
        ADAPT_QUALITY_RHO
        * state.quality_ema
        + (
            1.0
            - ADAPT_QUALITY_RHO
        )
        * float(current_quality)
    )

    improvement_ema = (
        ADAPT_IMPROVEMENT_RHO
        * state.improvement_ema
        + (
            1.0
            - ADAPT_IMPROVEMENT_RHO
        )
        * improvement
    )

    uncertainty_ema = (
        ADAPT_UNCERTAINTY_RHO
        * state.uncertainty_ema
        + (
            1.0
            - ADAPT_UNCERTAINTY_RHO
        )
        * float(uncertainty)
    )

    shortfall = max(
        0.0,
        float(target_refresh_ratio)
        - float(actual_refresh_ratio),
    )
    fairness_debt = clamp01(
        ADAPT_FAIRNESS_RHO
        * state.fairness_debt
        + (
            1.0
            - ADAPT_FAIRNESS_RHO
        )
        * shortfall
    )

    return AdaptiveClientState(
        quality_ema=clamp01(
            quality_ema
        ),
        improvement_ema=float(
            improvement_ema
        ),
        uncertainty_ema=clamp01(
            uncertainty_ema
        ),
        fairness_debt=fairness_debt,
        last_quality=float(
            current_quality
        ),
    )


ADAPTIVE_REFRESH_CONFIG = AdaptiveRefreshConfig()


# Integrated training, privacy, aggregation, and communication helpers

In [21]:

# ============================================================
# Integrated local training, QOTPH-aware adaptive LazyQ,
# resource-compatible sparse communication, DP, CKKS,
# server EMA, and operational layer sparing
# ============================================================

def clone_state_dict(
    state_dict: Dict[str, torch.Tensor],
) -> OrderedDict:
    return OrderedDict(
        (
            name,
            tensor.detach().cpu().clone(),
        )
        for name, tensor in state_dict.items()
    )


def is_quantum_coupled_name(
    name: str,
) -> bool:
    return (
        name.startswith("qbank.adapters.")
        or name.startswith("qbank.q_weights.")
        or name.startswith("qnn.")
    )


def _adapter_circuit_id(
    name: str,
) -> Optional[int]:
    if not name.startswith(
        "qbank.adapters."
    ):
        return None

    parts = name.split(".")
    if len(parts) < 3:
        return None
    return int(parts[2])


def _shared_quantum_weight_id(
    name: str,
) -> Optional[int]:
    if not name.startswith(
        "qbank.q_weights."
    ):
        return None

    parts = name.split(".")
    if len(parts) < 3:
        return None
    return int(parts[2])


def is_quantum_parameter_active(
    name: str,
    active_circuit_ids: Optional[
        Tuple[int, ...]
    ],
) -> bool:
    if active_circuit_ids is None:
        return True

    active_set = set(
        int(cid)
        for cid in active_circuit_ids
    )

    adapter_id = _adapter_circuit_id(
        name
    )
    if adapter_id is not None:
        return adapter_id in active_set

    weight_id = _shared_quantum_weight_id(
        name
    )
    if weight_id is not None:
        active_weight_ids = {
            circuit_id % R_UNIQUE
            for circuit_id in active_set
        }
        return weight_id in active_weight_ids

    return name.startswith("qnn.")


def parameter_block_name(
    name: str,
) -> str:
    return name.rsplit(
        ".",
        1,
    )[0]


def set_trainability_from_mask(
    model: nn.Module,
    active_mask: Dict[str, bool],
) -> None:
    for name, parameter in model.named_parameters():
        parameter.requires_grad = (
            True
            if is_quantum_coupled_name(name)
            else active_mask.get(
                name,
                True,
            )
        )


def split_parameter_groups(
    model: nn.Module,
    active_circuit_ids: Optional[
        Tuple[int, ...]
    ] = None,
):
    classical_named = []
    quantum_named = []

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue

        if is_quantum_coupled_name(name):
            if is_quantum_parameter_active(
                name,
                active_circuit_ids,
            ):
                quantum_named.append(
                    (
                        name,
                        parameter,
                    )
                )
        else:
            classical_named.append(
                (
                    name,
                    parameter,
                )
            )

    return (
        classical_named,
        quantum_named,
    )


def _build_optimizer(
    named_parameters,
    optimizer_name: str,
    learning_rate: float,
):
    parameters = [
        parameter
        for _, parameter in named_parameters
    ]

    if not parameters:
        return None

    optimizer_name = (
        optimizer_name.lower()
    )

    if optimizer_name == "adam":
        return torch.optim.Adam(
            parameters,
            lr=learning_rate,
        )

    if optimizer_name == "sgd":
        return torch.optim.SGD(
            parameters,
            lr=learning_rate,
        )

    raise ValueError(
        "optimizer_name must be 'adam' or 'sgd'"
    )


def exact_evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: nn.Module,
    device: torch.device,
):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    y_true = []
    y_pred = []
    y_proba = []
    softmax = nn.Softmax(dim=1)

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            logits = model(images)
            batch_size_now = labels.size(0)

            total_loss += (
                loss_fn(
                    logits,
                    labels,
                ).item()
                * batch_size_now
            )
            predictions = logits.argmax(
                dim=1
            )

            total_correct += (
                predictions == labels
            ).sum().item()
            total_examples += batch_size_now

            y_true.extend(
                labels.cpu().tolist()
            )
            y_pred.extend(
                predictions.cpu().tolist()
            )
            y_proba.extend(
                softmax(logits).cpu().numpy()
            )

    return (
        total_loss
        / max(total_examples, 1),
        100.0
        * total_correct
        / max(total_examples, 1),
        y_pred,
        y_true,
        np.asarray(y_proba),
    )


def _reference_on_device(
    reference_state: Dict[str, torch.Tensor],
    model: nn.Module,
) -> Dict[str, torch.Tensor]:
    return {
        name: reference_state[name].to(
            parameter.device,
            dtype=parameter.dtype,
        )
        for name, parameter in model.named_parameters()
    }


def _proximal_penalty(
    named_parameters,
    reference: Dict[str, torch.Tensor],
    mu: float,
) -> torch.Tensor:
    if not named_parameters or mu <= 0:
        if named_parameters:
            return named_parameters[0][
                1
            ].new_zeros(())
        return torch.tensor(
            0.0,
            device=DEVICE,
        )

    penalty = named_parameters[0][
        1
    ].new_zeros(())

    for name, parameter in named_parameters:
        penalty = penalty + torch.sum(
            (
                parameter
                - reference[name]
            )
            ** 2
        )

    return 0.5 * mu * penalty


def _flat_parameters(
    named_parameters,
) -> torch.Tensor:
    if not named_parameters:
        return torch.empty(
            0,
            device=DEVICE,
        )

    return torch.cat(
        [
            parameter.detach().reshape(-1)
            for _, parameter in named_parameters
        ]
    )


def train_local_standard(
    model: nn.Module,
    trainloader: DataLoader,
    reference_state: Dict[str, torch.Tensor],
    local_epochs: int,
    device: torch.device,
    **_,
):
    model.train()
    criterion = nn.CrossEntropyLoss()

    active_named = [
        (
            name,
            parameter,
        )
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]

    optimizer = _build_optimizer(
        active_named,
        optimizer_name="adam",
        learning_rate=LR_CLASSICAL,
    )
    reference = _reference_on_device(
        reference_state,
        model,
    )

    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    local_steps = 0

    for _epoch in range(local_epochs):
        for images, labels in trainloader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )
            logits = model(images)
            task_loss = criterion(
                logits,
                labels,
            )

            if USE_FEDPROX:
                prox = _proximal_penalty(
                    active_named,
                    reference,
                    FEDPROX_MU,
                )
            else:
                prox = task_loss.new_zeros(
                    ()
                )

            (
                task_loss
                + prox
            ).backward()
            optimizer.step()

            total_loss += (
                task_loss.item()
                * labels.size(0)
            )
            total_correct += (
                logits.argmax(dim=1)
                == labels
            ).sum().item()
            total_examples += labels.size(
                0
            )
            local_steps += 1

    return {
        "train_loss": (
            total_loss
            / max(total_examples, 1)
        ),
        "train_acc": (
            100.0
            * total_correct
            / max(total_examples, 1)
        ),
        "quantum_forward_passes": int(
            local_steps
        ),
        "active_quantum_circuit_forwards": int(
            local_steps
        ),
        "fresh_quantum_backward_passes": int(
            local_steps
        ),
        "lazy_quantum_steps": 0,
        "refresh_ratio": 1.0,
        "target_refresh_ratio": 1.0,
        "max_quantum_drift": 0.0,
        "local_steps": int(
            local_steps
        ),
        "refresh_reason_counts": {
            "naive": int(local_steps)
        },
        "qotph_runtime": {},
    }


def _batch_active_circuits(
    policy_active_circuits: Tuple[int, ...],
) -> Tuple[int, ...]:
    if (
        CIRCUIT_DROPOUT_PROB <= 0
        or len(policy_active_circuits) <= 1
    ):
        return policy_active_circuits

    retained = [
        circuit_id
        for circuit_id in policy_active_circuits
        if random.random()
        >= CIRCUIT_DROPOUT_PROB
    ]

    if not retained:
        retained = [
            random.choice(
                policy_active_circuits
            )
        ]

    return tuple(
        sorted(retained)
    )


def train_local_parallel_qbank(
    model: ParallelQBankAdeptNet,
    trainloader: DataLoader,
    reference_state: Dict[str, torch.Tensor],
    local_epochs: int,
    device: torch.device,
    adaptive_state: Optional[
        AdaptiveClientState
    ] = None,
    resource_profile: Optional[
        ClientResourceProfile
    ] = None,
    execution_policy: Optional[
        ClientExecutionPolicy
    ] = None,
):
    """
    Fresh steps construct the complete protected quantum autograd path.
    Lazy steps keep the classical path trainable while reusing a decayed
    EMA of the most recent quantum-coupled task gradient.

    In adaptive mode, quality, learning stagnation, uncertainty, drift,
    fairness debt, and client resource capacity determine the target
    refresh interval. Warm-up, excessive drift, maximum staleness, and
    a minimum refresh budget always override the normal policy.
    """
    model.train()
    criterion = nn.CrossEntropyLoss()
    reference = _reference_on_device(
        reference_state,
        model,
    )

    if execution_policy is None:
        execution_policy = ClientExecutionPolicy(
            local_epochs=local_epochs,
            active_circuit_ids=tuple(
                range(K_CIRCUITS)
            ),
            maximum_refresh_ratio=(
                MAX_REFRESH_RATIO
            ),
            upload_fraction=1.0,
            capacity_score=1.0,
        )

    if adaptive_state is None:
        adaptive_state = AdaptiveClientState()

    if resource_profile is None:
        resource_profile = ClientResourceProfile(
            tier="default",
            compute_score=1.0,
            memory_score=1.0,
            battery_score=1.0,
            qpu_latency_score=1.0,
            bandwidth_score=1.0,
        )

    policy_circuits = tuple(
        execution_policy.active_circuit_ids
    )

    classical_named, quantum_named = (
        split_parameter_groups(
            model,
            active_circuit_ids=policy_circuits,
        )
    )

    classical_optimizer = _build_optimizer(
        classical_named,
        optimizer_name="adam",
        learning_rate=LR_CLASSICAL,
    )
    quantum_optimizer = _build_optimizer(
        quantum_named,
        optimizer_name="sgd",
        learning_rate=LR_QUANTUM,
    )

    if quantum_optimizer is None:
        raise RuntimeError(
            "No active quantum-coupled parameters "
            "were selected for this client"
        )

    anchor = _flat_parameters(
        quantum_named
    ).clone()
    ema_task_gradients = None

    if QUANTUM_TRAINING == "lazy":
        last_refresh_step = -REFRESH_EVERY
    else:
        last_refresh_step = -MAX_STALE_STEPS

    local_step = 0
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    model_level_quantum_forwards = 0
    active_quantum_circuit_forwards = 0
    fresh_quantum_backward_passes = 0
    lazy_quantum_steps = 0
    max_quantum_drift = 0.0

    refresh_reason_counts = defaultdict(
        int
    )
    target_refresh_ratios = []
    learning_need_values = []
    capacity_values = []

    reset_qotph_runtime_stats()

    for _epoch in range(local_epochs):
        for images, labels in trainloader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            current_q = _flat_parameters(
                quantum_named
            )
            drift = torch.linalg.vector_norm(
                current_q
                - anchor
            ).item()
            max_quantum_drift = max(
                max_quantum_drift,
                drift,
            )

            if QUANTUM_TRAINING == "naive":
                do_refresh = True
                refresh_reason = "naive"
                target_ratio = 1.0
                learning_need = 1.0
                capacity = (
                    execution_policy.capacity_score
                )

            elif QUANTUM_TRAINING == "lazy":
                threshold_hit = (
                    drift > TAU_DRIFT
                )
                cache_missing = (
                    ema_task_gradients is None
                )
                interval_hit = (
                    local_step
                    - last_refresh_step
                ) >= REFRESH_EVERY

                do_refresh = (
                    local_step == 0
                    or threshold_hit
                    or cache_missing
                    or interval_hit
                )

                if local_step == 0:
                    refresh_reason = "first_step"
                elif threshold_hit:
                    refresh_reason = "drift"
                elif cache_missing:
                    refresh_reason = "cache_missing"
                elif interval_hit:
                    refresh_reason = "fixed_interval"
                else:
                    refresh_reason = "cached_gradient"

                target_ratio = min(
                    1.0,
                    1.0
                    / max(
                        REFRESH_EVERY,
                        1,
                    ),
                )
                learning_need = float(
                    1.0
                    - adaptive_state.quality_ema
                )
                capacity = (
                    execution_policy.capacity_score
                )

            elif QUANTUM_TRAINING == "adaptive_lazy":
                normalized_drift = clamp01(
                    drift
                    / max(
                        TAU_DRIFT,
                        1e-12,
                    )
                )

                (
                    adaptive_interval,
                    target_ratio,
                    learning_need,
                    capacity,
                ) = adaptive_refresh_interval(
                    state=adaptive_state,
                    profile=resource_profile,
                    normalized_drift=normalized_drift,
                    config=ADAPTIVE_REFRESH_CONFIG,
                    maximum_ratio_override=(
                        execution_policy.maximum_refresh_ratio
                    ),
                )

                (
                    do_refresh,
                    refresh_reason,
                ) = should_refresh_quantum_backward(
                    local_step=local_step,
                    last_refresh_step=(
                        last_refresh_step
                    ),
                    fresh_count=(
                        fresh_quantum_backward_passes
                    ),
                    drift=drift,
                    interval=adaptive_interval,
                    config=ADAPTIVE_REFRESH_CONFIG,
                )

                if ema_task_gradients is None:
                    do_refresh = True
                    refresh_reason = "cache_missing"

            else:
                raise ValueError(
                    f"Unsupported QUANTUM_TRAINING="
                    f"{QUANTUM_TRAINING!r}"
                )

            target_refresh_ratios.append(
                float(target_ratio)
            )
            learning_need_values.append(
                float(learning_need)
            )
            capacity_values.append(
                float(capacity)
            )

            batch_circuits = (
                _batch_active_circuits(
                    policy_circuits
                )
            )

            if do_refresh:
                refresh_reason_counts[
                    refresh_reason
                ] += 1

                if classical_optimizer is not None:
                    classical_optimizer.zero_grad(
                        set_to_none=True
                    )
                quantum_optimizer.zero_grad(
                    set_to_none=True
                )

                latent = model.encode(images)
                q_features = model.quantum_features(
                    latent,
                    detach_circuit=False,
                    active_circuit_ids=(
                        batch_circuits
                    ),
                )
                model_level_quantum_forwards += 1
                active_quantum_circuit_forwards += len(
                    batch_circuits
                )

                logits = model.classify(
                    latent,
                    q_features,
                )
                task_loss = criterion(
                    logits,
                    labels,
                )

                if USE_FEDPROX:
                    all_active_named = (
                        classical_named
                        + quantum_named
                    )
                    prox = _proximal_penalty(
                        all_active_named,
                        reference,
                        FEDPROX_MU,
                    )
                else:
                    prox = task_loss.new_zeros(
                        ()
                    )

                (
                    task_loss
                    + prox
                ).backward()

                fresh_task_gradients = []

                for name, parameter in quantum_named:
                    if parameter.grad is None:
                        total_gradient = (
                            torch.zeros_like(
                                parameter
                            )
                        )
                    else:
                        total_gradient = (
                            parameter.grad.detach().clone()
                        )

                    if USE_FEDPROX:
                        prox_gradient = (
                            FEDPROX_MU
                            * (
                                parameter.detach()
                                - reference[name]
                            )
                        )
                    else:
                        prox_gradient = (
                            torch.zeros_like(
                                parameter
                            )
                        )

                    fresh_task_gradients.append(
                        total_gradient
                        - prox_gradient
                    )

                if ema_task_gradients is None:
                    ema_task_gradients = [
                        gradient.clone()
                        for gradient in fresh_task_gradients
                    ]
                else:
                    ema_task_gradients = [
                        BETA_QGRAD
                        * old_gradient
                        + (
                            1.0
                            - BETA_QGRAD
                        )
                        * fresh_gradient
                        for (
                            old_gradient,
                            fresh_gradient,
                        ) in zip(
                            ema_task_gradients,
                            fresh_task_gradients,
                        )
                    ]

                if classical_optimizer is not None:
                    classical_optimizer.step()
                quantum_optimizer.step()

                anchor = _flat_parameters(
                    quantum_named
                ).clone()
                last_refresh_step = local_step
                fresh_quantum_backward_passes += 1

            else:
                refresh_reason_counts[
                    "cached_gradient"
                ] += 1

                if classical_optimizer is not None:
                    classical_optimizer.zero_grad(
                        set_to_none=True
                    )

                latent = model.encode(images)
                q_features = model.quantum_features(
                    latent,
                    detach_circuit=True,
                    active_circuit_ids=(
                        batch_circuits
                    ),
                )
                model_level_quantum_forwards += 1
                active_quantum_circuit_forwards += len(
                    batch_circuits
                )

                logits = model.classify(
                    latent,
                    q_features,
                )
                task_loss = criterion(
                    logits,
                    labels,
                )

                if (
                    USE_FEDPROX
                    and classical_named
                ):
                    prox_classical = (
                        _proximal_penalty(
                            classical_named,
                            reference,
                            FEDPROX_MU,
                        )
                    )
                else:
                    prox_classical = (
                        task_loss.new_zeros(
                            ()
                        )
                    )

                classical_objective = (
                    task_loss
                    + prox_classical
                )

                if (
                    classical_optimizer is not None
                    and classical_objective.requires_grad
                ):
                    classical_objective.backward()
                    classical_optimizer.step()

                quantum_optimizer.zero_grad(
                    set_to_none=True
                )

                age = max(
                    local_step
                    - last_refresh_step,
                    0,
                )
                stale_scale = (
                    GAMMA_STALE
                    ** age
                )

                for (
                    (
                        name,
                        parameter,
                    ),
                    cached_task_gradient,
                ) in zip(
                    quantum_named,
                    ema_task_gradients,
                ):
                    gradient = (
                        stale_scale
                        * cached_task_gradient
                    )

                    if USE_FEDPROX:
                        gradient = (
                            gradient
                            + FEDPROX_MU
                            * (
                                parameter.detach()
                                - reference[name]
                            )
                        )

                    parameter.grad = (
                        gradient.detach().clone()
                    )

                quantum_optimizer.step()
                lazy_quantum_steps += 1

            total_loss += (
                task_loss.item()
                * labels.size(0)
            )
            total_correct += (
                logits.argmax(dim=1)
                == labels
            ).sum().item()
            total_examples += labels.size(
                0
            )
            local_step += 1

    actual_refresh_ratio = (
        fresh_quantum_backward_passes
        / max(local_step, 1)
    )

    return {
        "train_loss": (
            total_loss
            / max(total_examples, 1)
        ),
        "train_acc": (
            100.0
            * total_correct
            / max(total_examples, 1)
        ),
        "quantum_forward_passes": int(
            model_level_quantum_forwards
        ),
        "active_quantum_circuit_forwards": int(
            active_quantum_circuit_forwards
        ),
        "fresh_quantum_backward_passes": int(
            fresh_quantum_backward_passes
        ),
        "lazy_quantum_steps": int(
            lazy_quantum_steps
        ),
        "refresh_ratio": float(
            actual_refresh_ratio
        ),
        "target_refresh_ratio": float(
            np.mean(target_refresh_ratios)
            if target_refresh_ratios
            else 1.0
        ),
        "mean_learning_need": float(
            np.mean(learning_need_values)
            if learning_need_values
            else 0.0
        ),
        "mean_capacity": float(
            np.mean(capacity_values)
            if capacity_values
            else 1.0
        ),
        "max_quantum_drift": float(
            max_quantum_drift
        ),
        "local_steps": int(
            local_step
        ),
        "refresh_reason_counts": dict(
            refresh_reason_counts
        ),
        "active_circuit_ids": list(
            policy_circuits
        ),
        "qotph_runtime": (
            snapshot_qotph_runtime_stats()
        ),
    }


def train_local_model(
    model: nn.Module,
    trainloader: DataLoader,
    reference_state: Dict[str, torch.Tensor],
    local_epochs: int,
    device: torch.device,
    adaptive_state: Optional[
        AdaptiveClientState
    ] = None,
    resource_profile: Optional[
        ClientResourceProfile
    ] = None,
    execution_policy: Optional[
        ClientExecutionPolicy
    ] = None,
):
    if isinstance(
        model,
        ParallelQBankAdeptNet,
    ):
        return train_local_parallel_qbank(
            model=model,
            trainloader=trainloader,
            reference_state=reference_state,
            local_epochs=local_epochs,
            device=device,
            adaptive_state=adaptive_state,
            resource_profile=resource_profile,
            execution_policy=execution_policy,
        )

    return train_local_standard(
        model=model,
        trainloader=trainloader,
        reference_state=reference_state,
        local_epochs=local_epochs,
        device=device,
    )


def initialize_he_contexts():
    secret_context = context()
    public_context = ts.context_from(
        secret_context.serialize(
            save_secret_key=False
        )
    )
    return (
        secret_context,
        public_context,
    )


def select_client_upload_names(
    model: nn.Module,
    active_mask: Dict[str, bool],
    active_circuit_ids: Tuple[int, ...],
    upload_fraction: float,
    importance_history: Dict[str, float],
    round_number: int,
) -> set:
    """
    Keep global tensor shapes compatible while permitting sparse uploads.

    Quantum parameters are transmitted only when their circuit/shared tensor
    was active. Classical layer sparing remains authoritative. Optional
    bandwidth sparsity selects the most important active classical blocks.
    """
    parameter_names = {
        name
        for name, _ in model.named_parameters()
    }

    quantum_names = {
        name
        for name in parameter_names
        if (
            is_quantum_coupled_name(name)
            and is_quantum_parameter_active(
                name,
                active_circuit_ids,
            )
        )
    }

    classical_names = {
        name
        for name in parameter_names
        if (
            not is_quantum_coupled_name(name)
            and active_mask.get(
                name,
                True,
            )
        )
    }

    if (
        not USE_BANDWIDTH_SPARSE_UPLOAD
        or round_number
        <= BANDWIDTH_WARMUP_ROUNDS
        or upload_fraction >= 0.999
    ):
        return (
            quantum_names
            | classical_names
        )

    mandatory_prefixes = (
        "classifier.",
        "fusion.",
    )
    mandatory_names = {
        name
        for name in classical_names
        if name.startswith(
            mandatory_prefixes
        )
    }

    optional_blocks = sorted(
        {
            parameter_block_name(name)
            for name in classical_names
            if name not in mandatory_names
        },
        key=lambda block: (
            -float(
                importance_history.get(
                    block,
                    float("inf"),
                )
                if importance_history.get(
                    block
                ) is not None
                else float("inf")
            ),
            block,
        ),
    )

    keep_count = int(
        math.ceil(
            upload_fraction
            * len(optional_blocks)
        )
    )
    selected_blocks = set(
        optional_blocks[:keep_count]
    )

    selected_classical = {
        name
        for name in classical_names
        if (
            name in mandatory_names
            or parameter_block_name(name)
            in selected_blocks
        )
    }

    return (
        quantum_names
        | selected_classical
    )


def _update_dp_arrays(
    model: nn.Module,
    reference_state: Dict[str, torch.Tensor],
    upload_names: set,
    dp_seed: int,
):
    parameters = dict(
        model.named_parameters()
    )

    raw_updates = {}
    norm_sq = 0.0

    for name in upload_names:
        if name not in parameters:
            continue

        tensor = parameters[name].detach().cpu()
        reference = reference_state[name].detach().cpu()
        update = (
            tensor
            - reference
        ).numpy()
        raw_updates[name] = update
        norm_sq += float(
            np.sum(
                update.astype(
                    np.float64
                )
                ** 2
            )
        )

    update_norm = math.sqrt(
        norm_sq
    )
    clip_factor = min(
        1.0,
        UPDATE_DP_CLIP_NORM
        / max(
            update_norm,
            1e-12,
        ),
    )

    rng = np.random.default_rng(
        dp_seed
    )
    private_arrays = {}

    for name, update in raw_updates.items():
        reference = (
            reference_state[name]
            .detach()
            .cpu()
            .numpy()
        )
        clipped_update = (
            update
            * clip_factor
        )

        if (
            UPDATE_DP_NOISE_MULTIPLIER
            > 0
        ):
            noise = rng.normal(
                loc=0.0,
                scale=(
                    UPDATE_DP_NOISE_MULTIPLIER
                    * UPDATE_DP_CLIP_NORM
                ),
                size=clipped_update.shape,
            ).astype(
                clipped_update.dtype
            )
        else:
            noise = np.zeros_like(
                clipped_update
            )

        private_arrays[name] = (
            reference
            + clipped_update
            + noise
        )

    return (
        private_arrays,
        {
            "update_norm": float(
                update_norm
            ),
            "clip_factor": float(
                clip_factor
            ),
            "noise_multiplier": float(
                UPDATE_DP_NOISE_MULTIPLIER
            ),
        },
    )


def build_client_payload(
    model: nn.Module,
    active_mask: Dict[str, bool],
    use_he: bool,
    public_context=None,
    reference_state: Optional[
        Dict[str, torch.Tensor]
    ] = None,
    upload_names: Optional[set] = None,
    dp_seed: int = 0,
):
    payload = {}
    total_bytes = 0

    if upload_names is None:
        upload_names = {
            name
            for name, _ in model.named_parameters()
            if (
                is_quantum_coupled_name(name)
                or active_mask.get(
                    name,
                    True,
                )
            )
        }

    if USE_UPDATE_DP:
        if reference_state is None:
            raise ValueError(
                "reference_state is required "
                "for update-level DP"
            )
        private_arrays, update_dp_stats = (
            _update_dp_arrays(
                model=model,
                reference_state=(
                    reference_state
                ),
                upload_names=upload_names,
                dp_seed=dp_seed,
            )
        )
    else:
        private_arrays = {}
        update_dp_stats = {}

    parameter_names = {
        name
        for name, _ in model.named_parameters()
    }

    for name, tensor in model.state_dict().items():
        if name not in upload_names:
            continue

        if (
            USE_UPDATE_DP
            and name in parameter_names
        ):
            array = np.asarray(
                private_arrays[name]
            )
        else:
            array = (
                tensor.detach()
                .cpu()
                .numpy()
            )

        if (
            use_he
            and name
            == ENCRYPTED_FINAL_WEIGHT
        ):
            if public_context is None:
                raise ValueError(
                    "public_context is required "
                    "when HE is enabled"
                )

            encrypted = ts.ckks_tensor(
                public_context,
                array,
            )
            serialized = (
                encrypted.serialize()
            )

            payload[name] = {
                "encrypted": True,
                "data": serialized,
                "shape": tuple(
                    array.shape
                ),
                "dtype": str(
                    array.dtype
                ),
            }
            total_bytes += len(
                serialized
            )

        else:
            payload[name] = {
                "encrypted": False,
                "data": array,
                "shape": tuple(
                    array.shape
                ),
                "dtype": str(
                    array.dtype
                ),
            }
            total_bytes += array.nbytes

    return (
        payload,
        total_bytes,
        update_dp_stats,
    )


def aggregate_client_payloads(
    client_payloads,
    aggregation_weights: List[float],
    previous_raw_state: Dict[str, torch.Tensor],
    use_he: bool,
    public_context=None,
    secret_context=None,
):
    if len(
        client_payloads
    ) != len(
        aggregation_weights
    ):
        raise ValueError(
            "Payload count and weight count "
            "do not match"
        )

    aggregated_state = clone_state_dict(
        previous_raw_state
    )
    transmitted_names = sorted(
        set().union(
            *(
                payload.keys()
                for payload in client_payloads
            )
        )
    )

    for name in transmitted_names:
        entries = [
            payload[name]
            for payload in client_payloads
            if name in payload
        ]
        entry_weights = [
            weight
            for payload, weight in zip(
                client_payloads,
                aggregation_weights,
            )
            if name in payload
        ]

        weight_sum = sum(
            entry_weights
        )
        if weight_sum <= 0:
            continue

        entry_weights = [
            weight
            / weight_sum
            for weight in entry_weights
        ]

        if entries[0]["encrypted"]:
            if (
                not use_he
                or public_context is None
                or secret_context is None
            ):
                raise ValueError(
                    "HE contexts are required "
                    "for encrypted aggregation"
                )

            encrypted_tensors = [
                ts.ckks_tensor_from(
                    public_context,
                    entry["data"],
                )
                for entry in entries
            ]

            encrypted_sum = (
                encrypted_tensors[0]
                * entry_weights[0]
            )

            for (
                encrypted_tensor,
                weight,
            ) in zip(
                encrypted_tensors[1:],
                entry_weights[1:],
            ):
                encrypted_sum = (
                    encrypted_sum
                    + encrypted_tensor
                    * weight
                )

            decrypted = np.asarray(
                encrypted_sum.decrypt(
                    secret_context.secret_key()
                ),
                dtype=np.float32,
            ).reshape(
                entries[0]["shape"]
            )

            aggregated_state[name] = (
                torch.as_tensor(
                    decrypted,
                    dtype=(
                        previous_raw_state[
                            name
                        ].dtype
                    ),
                )
            )

        else:
            weighted_array = np.zeros(
                entries[0]["shape"],
                dtype=np.float64,
            )

            for (
                entry,
                weight,
            ) in zip(
                entries,
                entry_weights,
            ):
                weighted_array += (
                    entry["data"].astype(
                        np.float64
                    )
                    * weight
                )

            aggregated_state[name] = (
                torch.as_tensor(
                    weighted_array,
                    dtype=(
                        previous_raw_state[
                            name
                        ].dtype
                    ),
                )
            )

    return aggregated_state


def apply_blockwise_server_ema(
    raw_state: Dict[str, torch.Tensor],
    previous_ema_state: Dict[str, torch.Tensor],
):
    ema_state = OrderedDict()

    for name, raw_tensor in raw_state.items():
        beta = (
            SERVER_EMA_BETA_QUANTUM
            if is_quantum_coupled_name(
                name
            )
            else SERVER_EMA_BETA_CLASSICAL
        )

        if (
            not USE_SERVER_EMA
            or beta <= 0
        ):
            ema_state[name] = (
                raw_tensor.detach()
                .cpu()
                .clone()
            )
        else:
            ema_state[name] = (
                beta
                * previous_ema_state[name]
                + (
                    1.0
                    - beta
                )
                * raw_tensor
            ).detach().cpu()

    return ema_state


def initialize_layer_sparing_state(
    model: nn.Module,
):
    active_mask = {
        name: True
        for name, _ in model.named_parameters()
    }

    blocks = {
        parameter_block_name(name)
        for name, _ in model.named_parameters()
        if not is_quantum_coupled_name(
            name
        )
    }

    importance_history = {
        block: None
        for block in blocks
    }
    low_importance_counts = {
        block: 0
        for block in blocks
    }
    frozen_blocks = set()

    return (
        active_mask,
        importance_history,
        low_importance_counts,
        frozen_blocks,
    )


def update_layer_sparing_mask(
    model: nn.Module,
    raw_state: Dict[str, torch.Tensor],
    previous_raw_state: Dict[str, torch.Tensor],
    active_mask: Dict[str, bool],
    importance_history: Dict[str, float],
    low_importance_counts: Dict[str, int],
    frozen_blocks: set,
    completed_round: int,
    transmitted_names: Optional[set] = None,
):
    if not USE_LAYER_SPARING:
        return (
            active_mask.copy(),
            importance_history,
            low_importance_counts,
            frozen_blocks,
            {},
        )

    block_differences = defaultdict(
        list
    )

    for name, _ in model.named_parameters():
        if is_quantum_coupled_name(
            name
        ):
            continue

        if (
            transmitted_names is not None
            and name not in transmitted_names
        ):
            continue

        block = parameter_block_name(
            name
        )
        difference = (
            raw_state[name]
            - previous_raw_state[name]
        ).reshape(-1)
        block_differences[
            block
        ].append(
            difference
        )

    current_norms = {}

    for (
        block,
        differences,
    ) in block_differences.items():
        flat_difference = torch.cat(
            differences
        )
        current_norms[block] = (
            torch.linalg.vector_norm(
                flat_difference
            ).item()
        )

        previous_importance = (
            importance_history.get(
                block
            )
        )

        if previous_importance is None:
            importance_history[
                block
            ] = current_norms[
                block
            ]
        else:
            importance_history[
                block
            ] = (
                IMPORTANCE_EMA_ALPHA
                * previous_importance
                + (
                    1.0
                    - IMPORTANCE_EMA_ALPHA
                )
                * current_norms[block]
            )

        if (
            completed_round
            >= FREEZE_WARMUP_ROUNDS
        ):
            if (
                importance_history[block]
                < FREEZE_THRESHOLD
            ):
                low_importance_counts[
                    block
                ] = (
                    low_importance_counts.get(
                        block,
                        0,
                    )
                    + 1
                )
            else:
                low_importance_counts[
                    block
                ] = 0

            if (
                low_importance_counts[
                    block
                ]
                >= FREEZE_PATIENCE
            ):
                frozen_blocks.add(
                    block
                )

    next_active_mask = {}

    for name, _ in model.named_parameters():
        if is_quantum_coupled_name(
            name
        ):
            next_active_mask[
                name
            ] = True
        else:
            next_active_mask[
                name
            ] = (
                parameter_block_name(
                    name
                )
                not in frozen_blocks
            )

    return (
        next_active_mask,
        importance_history,
        low_importance_counts,
        frozen_blocks,
        current_norms,
    )


def full_model_parameter_bytes(
    model: nn.Module,
) -> int:
    return sum(
        parameter.numel()
        * parameter.element_size()
        for parameter in model.parameters()
    )


def select_round_clients(
    rng: np.random.Generator,
    num_clients: int,
    fraction: float,
    minimum: int,
) -> List[int]:
    selected_count = max(
        minimum,
        int(
            math.ceil(
                fraction
                * num_clients
            )
        ),
    )
    selected_count = min(
        selected_count,
        num_clients,
    )

    selected = rng.choice(
        num_clients,
        size=selected_count,
        replace=False,
    )

    return sorted(
        int(cid)
        for cid in selected.tolist()
    )


# Main Stage-2 federated experiment

In [22]:

# ============================================================
# Stage-2 federated experiment:
# QOTPH + adaptive LazyQ + heterogeneous resources + DP/CKKS
# ============================================================

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 96)
print("QOTPH + Adaptive LazyQ + Resource-Aware FL configuration")
print(f"Run profile:             {RUN_PROFILE}")
print(f"Device:                  {DEVICE}")
print(f"Model variant:           {MODEL_VARIANT}")
print(f"QOTPH mode:              {QOTPH_MODE}")
print(f"QOTPH execution:         {QOTPH_EXECUTION}")
print(f"QOTPH shots:             {QOTPH_SHOTS}")
print(f"Quantum training:        {QUANTUM_TRAINING}")
print(f"Partition mode:          {PARTITION_MODE}")

if PARTITION_MODE == "dirichlet":
    print(f"Dirichlet alpha:         {DIRICHLET_ALPHA}")
elif PARTITION_MODE == "balanced_rotating":
    print(
        "Balanced samples:        "
        f"{BALANCED_SAMPLES_PER_CLASS_PER_CLIENT}"
        "/class/client/round"
    )
elif PARTITION_MODE == "balanced_static_full":
    print("Static full-data split:  enabled")

print(f"Clients:                 {number_clients}")
print(f"Client fraction/round:   {frac_fit}")
print(f"Base local epochs:       {max_epochs}")
print(f"Aggregation mode:        {AGGREGATION_MODE}")
print(f"FedProx:                 {USE_FEDPROX} (mu={FEDPROX_MU})")
print(f"Resource-aware policy:   {USE_RESOURCE_AWARE_POLICY}")
print(f"Circuit masking:         {USE_CIRCUIT_MASKING}")
print(f"Bandwidth sparse upload: {USE_BANDWIDTH_SPARSE_UPLOAD}")
print(f"Quality DP:              {use_dp}")
print(f"Update DP:               {USE_UPDATE_DP}")
print(f"Selective CKKS:          {he}")
print("=" * 96)

if he:
    (
        secret_context,
        public_context,
    ) = initialize_he_contexts()
else:
    secret_context = None
    public_context = None

if PARTITION_MODE == "balanced_rotating":
    (
        balanced_scheduler,
        valloaders,
        testloader,
        partition_stats,
    ) = load_balanced_rotating_datasets(
        num_clients=number_clients,
        batch_size=batch_size,
        seed=seed,
        num_workers=num_workers,
        dataset=dataset,
        data_path=data_path,
        samples_per_class_per_client=(
            BALANCED_SAMPLES_PER_CLASS_PER_CLIENT
        ),
        val_per_class_per_client=(
            BALANCED_VAL_PER_CLASS_PER_CLIENT
        ),
        reuse_scope=BALANCED_REUSE_SCOPE,
    )
    trainloaders = None
else:
    balanced_scheduler = None
    (
        trainloaders,
        valloaders,
        testloader,
        partition_stats,
    ) = load_datasets(
        num_clients=number_clients,
        batch_size=batch_size,
        resize=None,
        seed=seed,
        num_workers=num_workers,
        splitter=splitter,
        dataset=dataset,
        data_path=data_path,
        partition_mode=PARTITION_MODE,
        dirichlet_alpha=DIRICHLET_ALPHA,
        min_client_samples=MIN_CLIENT_SAMPLES,
        balanced_val_per_class_per_client=(
            BALANCED_VAL_PER_CLASS_PER_CLIENT
        ),
    )

global_model = build_model(
    num_classes=len(CLASSES)
).to(DEVICE)

if isinstance(
    global_model,
    ParallelQBankAdeptNet,
):
    global_model.set_execution_context(
        client_id=-1,
        round_number=0,
        active_circuit_ids=tuple(
            range(K_CIRCUITS)
        ),
    )

criterion = nn.CrossEntropyLoss()

initial_state = clone_state_dict(
    global_model.state_dict()
)
previous_raw_state = clone_state_dict(
    initial_state
)
server_ema_state = clone_state_dict(
    initial_state
)
broadcast_state = clone_state_dict(
    initial_state
)

(
    active_mask,
    importance_history,
    low_importance_counts,
    frozen_blocks,
) = initialize_layer_sparing_state(
    global_model
)

quality_ema = {
    cid: None
    for cid in range(number_clients)
}

adaptive_client_states = {
    cid: AdaptiveClientState()
    for cid in range(number_clients)
}

resource_profiles = (
    build_client_resource_profiles(
        number_clients,
        RESOURCE_PROFILE_SEED,
    )
)

print("\nClient resource profiles")
for cid, profile in resource_profiles.items():
    print(
        f"Client {cid:02d} | tier={profile.tier:6s} | "
        f"compute={profile.compute_score:.2f} | "
        f"memory={profile.memory_score:.2f} | "
        f"battery={profile.battery_score:.2f} | "
        f"qpu={profile.qpu_latency_score:.2f} | "
        f"bandwidth={profile.bandwidth_score:.2f}"
    )

client_rng = np.random.default_rng(
    seed + 2026
)
full_bytes_per_client = (
    full_model_parameter_bytes(
        global_model
    )
)
round_history = []

print(
    f"\nGlobal model parameters: "
    f"{sum(parameter.numel() for parameter in global_model.parameters()):,}"
)
print(
    "Plain full-model payload/client: "
    f"{full_bytes_per_client / 1e6:.3f} MB"
)


def client_train_one_round(
    cid: int,
    round_number: int,
    round_trainloader: DataLoader,
    current_broadcast_state: Dict[str, torch.Tensor],
    current_active_mask: Dict[str, bool],
):
    profile = resource_profiles[cid]
    policy = allocate_client_policy(
        profile=profile,
        client_id=cid,
        round_number=round_number,
        base_local_epochs=max_epochs,
        k_circuits=K_CIRCUITS,
    )

    local_model = build_model(
        num_classes=len(CLASSES)
    ).to(DEVICE)
    local_model.load_state_dict(
        current_broadcast_state,
        strict=True,
    )
    set_trainability_from_mask(
        local_model,
        current_active_mask,
    )

    if isinstance(
        local_model,
        ParallelQBankAdeptNet,
    ):
        local_model.set_execution_context(
            client_id=cid,
            round_number=round_number,
            active_circuit_ids=(
                policy.active_circuit_ids
            ),
        )

    client_training_t0 = (
        time.perf_counter()
    )

    local_metrics = train_local_model(
        model=local_model,
        trainloader=round_trainloader,
        reference_state=(
            current_broadcast_state
        ),
        local_epochs=policy.local_epochs,
        device=DEVICE,
        adaptive_state=(
            adaptive_client_states[cid]
        ),
        resource_profile=profile,
        execution_policy=policy,
    )

    client_training_seconds = (
        time.perf_counter()
        - client_training_t0
    )

    if not isinstance(
        local_metrics,
        dict,
    ):
        raise TypeError(
            "Local training must return a metric dictionary"
        )

    (
        val_loss,
        val_acc_percent,
        _val_predictions,
        _val_targets,
        val_probabilities,
    ) = exact_evaluate(
        local_model,
        valloaders[cid],
        criterion,
        DEVICE,
    )

    true_val_acc = (
        val_acc_percent
        / 100.0
    )
    n_val = len(
        valloaders[cid].dataset
    )
    uncertainty = (
        normalized_predictive_entropy(
            val_probabilities
        )
    )

    if use_dp:
        transmitted_quality = (
            privatize_accuracy(
                true_val_acc,
                n_val,
                ε=DP_EPSILON,
            )
        )
    else:
        transmitted_quality = (
            true_val_acc
        )

    updated_adaptive_state = (
        update_adaptive_client_state(
            state=adaptive_client_states[
                cid
            ],
            current_quality=true_val_acc,
            uncertainty=uncertainty,
            actual_refresh_ratio=(
                local_metrics[
                    "refresh_ratio"
                ]
            ),
            target_refresh_ratio=(
                local_metrics[
                    "target_refresh_ratio"
                ]
            ),
        )
    )

    upload_names = (
        select_client_upload_names(
            model=local_model,
            active_mask=(
                current_active_mask
            ),
            active_circuit_ids=(
                policy.active_circuit_ids
            ),
            upload_fraction=(
                policy.upload_fraction
            ),
            importance_history=(
                importance_history
            ),
            round_number=round_number,
        )
    )

    (
        payload,
        payload_bytes,
        update_dp_stats,
    ) = build_client_payload(
        model=local_model,
        active_mask=current_active_mask,
        use_he=he,
        public_context=public_context,
        reference_state=(
            current_broadcast_state
        ),
        upload_names=upload_names,
        dp_seed=_stable_seed(
            seed,
            round_number,
            cid,
            "update_dp",
        ),
    )

    return {
        "cid": int(cid),
        "profile": asdict(profile),
        "policy": asdict(policy),
        "payload": payload,
        "payload_bytes": int(
            payload_bytes
        ),
        "upload_tensor_count": int(
            len(payload)
        ),
        "num_train_examples": int(
            len(round_trainloader.dataset)
        ),
        "true_val_acc": float(
            true_val_acc
        ),
        "transmitted_quality": float(
            transmitted_quality
        ),
        "val_loss": float(
            val_loss
        ),
        "uncertainty": float(
            uncertainty
        ),
        "local_metrics": local_metrics,
        "client_training_seconds": float(
            client_training_seconds
        ),
        "updated_adaptive_state": (
            updated_adaptive_state
        ),
        "update_dp_stats": (
            update_dp_stats
        ),
    }


print(f"\nTraining on {DEVICE}")
start_simulation = time.perf_counter()

for round_index in range(rounds):
    round_number = round_index + 1
    round_t0 = time.perf_counter()

    selected_clients = select_round_clients(
        client_rng,
        num_clients=number_clients,
        fraction=frac_fit,
        minimum=min_fit_clients,
    )

    if PARTITION_MODE == "balanced_rotating":
        (
            round_trainloaders,
            round_data_stats,
        ) = balanced_scheduler.get_round_loaders(
            round_index=round_index,
            selected_clients=selected_clients,
        )
    else:
        round_trainloaders = {
            cid: trainloaders[cid]
            for cid in selected_clients
        }
        round_data_stats = {}

    print("\n" + "#" * 96)
    print(
        f"Round {round_number}/{rounds}"
    )
    print(
        f"Selected clients: {selected_clients}"
    )
    print(
        "Frozen blocks applied before training/upload: "
        f"{sorted(frozen_blocks)}"
    )
    print("#" * 96)

    if PARTITION_MODE == "balanced_rotating":
        for cid in selected_clients:
            stat = round_data_stats[cid]
            print(
                f"Client {cid:02d} | "
                f"total={stat['total']:5d} | "
                f"block={stat['block_id']:02d} | "
                f"class_counts={stat['class_counts']}"
            )

    client_records = []

    for cid in selected_clients:
        print(
            f"[Client {cid}] local training started"
        )

        record = client_train_one_round(
            cid=cid,
            round_number=round_number,
            round_trainloader=(
                round_trainloaders[cid]
            ),
            current_broadcast_state=(
                broadcast_state
            ),
            current_active_mask=(
                active_mask
            ),
        )

        client_records.append(record)
        adaptive_client_states[cid] = (
            record[
                "updated_adaptive_state"
            ]
        )

        metrics = record[
            "local_metrics"
        ]
        policy = record["policy"]

        print(
            f"[Client {cid}] "
            f"tier={record['profile']['tier']} | "
            f"epochs={policy['local_epochs']} | "
            f"circuits={policy['active_circuit_ids']} | "
            f"train_acc={metrics['train_acc']:.2f}% | "
            f"val_acc={100.0 * record['true_val_acc']:.2f}% | "
            f"quality={record['transmitted_quality']:.4f} | "
            f"fresh_q={metrics['fresh_quantum_backward_passes']} | "
            f"lazy={metrics['lazy_quantum_steps']} | "
            f"refresh={metrics['refresh_ratio']:.4f} | "
            f"target={metrics['target_refresh_ratio']:.4f} | "
            f"drift={metrics['max_quantum_drift']:.5f} | "
            f"uncertainty={record['uncertainty']:.4f} | "
            f"time={record['client_training_seconds']:.2f}s | "
            f"upload={record['payload_bytes'] / 1e6:.3f} MB"
        )

        print(
            f"             refresh reasons="
            f"{metrics['refresh_reason_counts']} | "
            f"QOTPH runtime={metrics['qotph_runtime']}"
        )

    selected_quality_values = []

    for record in client_records:
        cid = record["cid"]
        new_quality = record[
            "transmitted_quality"
        ]

        if (
            quality_ema[cid] is None
            or not USE_QUALITY_EMA
        ):
            quality_ema[cid] = (
                new_quality
            )
        else:
            quality_ema[cid] = (
                QUALITY_EMA_RHO
                * quality_ema[cid]
                + (
                    1.0
                    - QUALITY_EMA_RHO
                )
                * new_quality
            )

        selected_quality_values.append(
            quality_ema[cid]
        )

    if AGGREGATION_MODE == "uniform":
        aggregation_weights = [
            1.0
            / len(client_records)
            for _record in client_records
        ]

    elif AGGREGATION_MODE == "sample_size":
        sample_counts = np.asarray(
            [
                record[
                    "num_train_examples"
                ]
                for record in client_records
            ],
            dtype=np.float64,
        )
        aggregation_weights = (
            sample_counts
            / sample_counts.sum()
        ).tolist()

    elif AGGREGATION_MODE == "quality":
        aggregation_weights = (
            accuracy_weights(
                selected_quality_values,
                τ=TAU_AGG,
            )
        )

    elif AGGREGATION_MODE == "hybrid":
        quality_component = np.asarray(
            accuracy_weights(
                selected_quality_values,
                τ=TAU_AGG,
            ),
            dtype=np.float64,
        )
        sample_counts = np.asarray(
            [
                record[
                    "num_train_examples"
                ]
                for record in client_records
            ],
            dtype=np.float64,
        )
        sample_component = (
            sample_counts
            / sample_counts.sum()
        )
        combined = (
            quality_component
            * sample_component
        )
        aggregation_weights = (
            combined
            / combined.sum()
        ).tolist()

    else:
        raise ValueError(
            "Unsupported AGGREGATION_MODE"
        )

    print(
        f"[Round {round_number}] "
        f"Smoothed qualities: "
        f"{np.round(selected_quality_values, 4).tolist()}"
    )
    print(
        f"[Round {round_number}] "
        f"{AGGREGATION_MODE} weights: "
        f"{np.round(aggregation_weights, 4).tolist()}"
    )

    raw_aggregated_state = (
        aggregate_client_payloads(
            client_payloads=[
                record["payload"]
                for record in client_records
            ],
            aggregation_weights=(
                aggregation_weights
            ),
            previous_raw_state=(
                previous_raw_state
            ),
            use_he=he,
            public_context=(
                public_context
            ),
            secret_context=(
                secret_context
            ),
        )
    )

    transmitted_parameter_names = set().union(
        *[
            record["payload"].keys()
            for record in client_records
        ]
    )

    (
        next_active_mask,
        importance_history,
        low_importance_counts,
        frozen_blocks,
        block_norms,
    ) = update_layer_sparing_mask(
        model=global_model,
        raw_state=(
            raw_aggregated_state
        ),
        previous_raw_state=(
            previous_raw_state
        ),
        active_mask=active_mask,
        importance_history=(
            importance_history
        ),
        low_importance_counts=(
            low_importance_counts
        ),
        frozen_blocks=frozen_blocks,
        completed_round=round_number,
        transmitted_names=(
            transmitted_parameter_names
        ),
    )

    server_ema_state = (
        apply_blockwise_server_ema(
            raw_state=(
                raw_aggregated_state
            ),
            previous_ema_state=(
                server_ema_state
            ),
        )
    )

    broadcast_state = (
        clone_state_dict(
            server_ema_state
        )
        if USE_SERVER_EMA
        else clone_state_dict(
            raw_aggregated_state
        )
    )

    global_model.load_state_dict(
        broadcast_state,
        strict=True,
    )

    if isinstance(
        global_model,
        ParallelQBankAdeptNet,
    ):
        global_model.set_execution_context(
            client_id=-1,
            round_number=round_number,
            active_circuit_ids=tuple(
                range(K_CIRCUITS)
            ),
        )

    (
        test_loss,
        test_acc,
        test_predictions,
        test_targets,
        test_probabilities,
    ) = exact_evaluate(
        global_model,
        testloader,
        criterion,
        DEVICE,
    )

    prediction_histogram = (
        np.bincount(
            np.asarray(
                test_predictions,
                dtype=np.int64,
            ),
            minlength=len(CLASSES),
        )
    )
    target_histogram = (
        np.bincount(
            np.asarray(
                test_targets,
                dtype=np.int64,
            ),
            minlength=len(CLASSES),
        )
    )

    actual_upload_bytes = sum(
        record["payload_bytes"]
        for record in client_records
    )
    full_upload_bytes = (
        full_bytes_per_client
        * len(selected_clients)
    )
    communication_reduction = (
        1.0
        - actual_upload_bytes
        / max(
            full_upload_bytes,
            1,
        )
    )

    total_fresh_q = sum(
        record["local_metrics"][
            "fresh_quantum_backward_passes"
        ]
        for record in client_records
    )
    total_lazy_q = sum(
        record["local_metrics"][
            "lazy_quantum_steps"
        ]
        for record in client_records
    )
    total_model_q_forwards = sum(
        record["local_metrics"][
            "quantum_forward_passes"
        ]
        for record in client_records
    )
    total_active_circuit_forwards = sum(
        record["local_metrics"][
            "active_quantum_circuit_forwards"
        ]
        for record in client_records
    )
    mean_refresh_ratio = float(
        np.mean(
            [
                record["local_metrics"][
                    "refresh_ratio"
                ]
                for record in client_records
            ]
        )
    )
    mean_target_refresh_ratio = float(
        np.mean(
            [
                record["local_metrics"][
                    "target_refresh_ratio"
                ]
                for record in client_records
            ]
        )
    )

    round_runtime_seconds = (
        time.perf_counter()
        - round_t0
    )
    total_client_training_seconds = sum(
        record[
            "client_training_seconds"
        ]
        for record in client_records
    )
    round_nontraining_seconds = max(
        0.0,
        round_runtime_seconds
        - total_client_training_seconds,
    )

    round_qotph_runtime = defaultdict(
        float
    )
    for record in client_records:
        for (
            key,
            value,
        ) in record["local_metrics"][
            "qotph_runtime"
        ].items():
            round_qotph_runtime[key] += float(
                value
            )

    client_metric_records = []

    for record in client_records:
        client_metric_records.append(
            {
                "cid": record["cid"],
                "profile": record["profile"],
                "policy": record["policy"],
                "num_train_examples": (
                    record[
                        "num_train_examples"
                    ]
                ),
                "train_acc": (
                    record["local_metrics"][
                        "train_acc"
                    ]
                ),
                "true_val_acc": (
                    record[
                        "true_val_acc"
                    ]
                ),
                "transmitted_quality": (
                    record[
                        "transmitted_quality"
                    ]
                ),
                "val_loss": (
                    record["val_loss"]
                ),
                "uncertainty": (
                    record[
                        "uncertainty"
                    ]
                ),
                "client_training_seconds": (
                    record[
                        "client_training_seconds"
                    ]
                ),
                "payload_bytes": (
                    record[
                        "payload_bytes"
                    ]
                ),
                "upload_tensor_count": (
                    record[
                        "upload_tensor_count"
                    ]
                ),
                "local_metrics": (
                    record[
                        "local_metrics"
                    ]
                ),
                "adaptive_state": asdict(
                    adaptive_client_states[
                        record["cid"]
                    ]
                ),
                "update_dp_stats": (
                    record[
                        "update_dp_stats"
                    ]
                ),
            }
        )

    round_summary = {
        "round": int(round_number),
        "selected_clients": (
            selected_clients
        ),
        "test_loss": float(
            test_loss
        ),
        "test_acc": float(
            test_acc
        ),
        "test_uncertainty": float(
            normalized_predictive_entropy(
                test_probabilities
            )
        ),
        "prediction_histogram": (
            prediction_histogram.tolist()
        ),
        "target_histogram": (
            target_histogram.tolist()
        ),
        "actual_upload_bytes": int(
            actual_upload_bytes
        ),
        "full_upload_bytes": int(
            full_upload_bytes
        ),
        "communication_reduction": float(
            communication_reduction
        ),
        "frozen_blocks": sorted(
            frozen_blocks
        ),
        "block_norms": (
            block_norms
        ),
        "fresh_quantum_backward_passes": int(
            total_fresh_q
        ),
        "lazy_quantum_steps": int(
            total_lazy_q
        ),
        "mean_refresh_ratio": float(
            mean_refresh_ratio
        ),
        "mean_target_refresh_ratio": float(
            mean_target_refresh_ratio
        ),
        "quantum_forward_passes": int(
            total_model_q_forwards
        ),
        "active_quantum_circuit_forwards": int(
            total_active_circuit_forwards
        ),
        "quality_values": (
            selected_quality_values
        ),
        "aggregation_weights": (
            aggregation_weights
        ),
        "round_runtime_seconds": float(
            round_runtime_seconds
        ),
        "client_training_seconds_total": float(
            total_client_training_seconds
        ),
        "round_nontraining_seconds": float(
            round_nontraining_seconds
        ),
        "qotph_runtime": dict(
            round_qotph_runtime
        ),
        "round_data_stats": (
            round_data_stats
        ),
        "client_records": (
            client_metric_records
        ),
    }
    round_history.append(
        round_summary
    )

    print(
        f"[Round {round_number}] "
        f"Global test loss={test_loss:.4f} | "
        f"accuracy={test_acc:.4f}%"
    )
    print(
        f"[Round {round_number}] "
        f"Prediction histogram: "
        f"{prediction_histogram.tolist()}"
    )
    print(
        f"[Round {round_number}] "
        f"Upload={actual_upload_bytes / 1e6:.3f} MB | "
        f"full={full_upload_bytes / 1e6:.3f} MB | "
        f"reduction={100.0 * communication_reduction:.2f}%"
    )
    print(
        f"[Round {round_number}] "
        f"fresh_q={total_fresh_q} | "
        f"lazy_q={total_lazy_q} | "
        f"refresh={mean_refresh_ratio:.4f} | "
        f"target={mean_target_refresh_ratio:.4f}"
    )
    print(
        f"[Round {round_number}] "
        f"model Q forwards={total_model_q_forwards} | "
        f"active circuit forwards={total_active_circuit_forwards}"
    )
    print(
        f"[Round {round_number}] "
        f"runtime={round_runtime_seconds:.2f}s | "
        f"client training={total_client_training_seconds:.2f}s | "
        f"other={round_nontraining_seconds:.2f}s"
    )
    print(
        f"[Round {round_number}] "
        f"QOTPH runtime={dict(round_qotph_runtime)}"
    )
    print(
        f"[Round {round_number}] "
        f"frozen blocks for next round="
        f"{sorted(frozen_blocks)}"
    )

    previous_raw_state = clone_state_dict(
        raw_aggregated_state
    )
    active_mask = next_active_mask

simulation_time = (
    time.perf_counter()
    - start_simulation
)

print("\n" + "=" * 96)
print(
    "Federated learning completed in "
    f"{simulation_time:.2f} seconds"
)
print(
    "Final test accuracy: "
    f"{round_history[-1]['test_acc']:.4f}%"
)
print("=" * 96)


QOTPH + Adaptive LazyQ + Resource-Aware FL configuration
Run profile:             stage2_learning_smoke
Device:                  cuda:0
Model variant:           parallel_qbank
QOTPH mode:              algorithm2
QOTPH execution:         analytic
QOTPH shots:             4096
Quantum training:        naive
Partition mode:          balanced_rotating
Balanced samples:        50/class/client/round
Clients:                 3
Client fraction/round:   1.0
Base local epochs:       3
Aggregation mode:        uniform
FedProx:                 False (mu=0.001)
Resource-aware policy:   False
Circuit masking:         False
Bandwidth sparse upload: False
Quality DP:              False
Update DP:               False
Selective CKKS:          False
Partition mode: balanced_rotating
Training samples/client/round: 50 per class (500 total)
Validation samples/client: 20 per class (200 total)
Reuse scope: per_client
Class blocks available: 118
Rotation stride: 5
Maximum no-repeat rounds for each client: 118


# Save the final model and Stage-2 ablation row

In [1]:

# ============================================================
# Save checkpoint, history, and a compact Stage-2 result row
# ============================================================

if save_results:
    os.makedirs(
        save_results,
        exist_ok=True,
    )

    total_runtime_seconds = float(
        simulation_time
    )

    total_fresh_quantum_backward_passes = int(
        sum(
            row[
                "fresh_quantum_backward_passes"
            ]
            for row in round_history
        )
    )
    total_lazy_quantum_steps = int(
        sum(
            row[
                "lazy_quantum_steps"
            ]
            for row in round_history
        )
    )
    total_quantum_steps = (
        total_fresh_quantum_backward_passes
        + total_lazy_quantum_steps
    )

    runtime_summary = {
        "total_runtime_seconds": (
            total_runtime_seconds
        ),
        "total_client_training_seconds": float(
            sum(
                row[
                    "client_training_seconds_total"
                ]
                for row in round_history
            )
        ),
        "total_quantum_forward_passes": int(
            sum(
                row[
                    "quantum_forward_passes"
                ]
                for row in round_history
            )
        ),
        "total_active_quantum_circuit_forwards": int(
            sum(
                row[
                    "active_quantum_circuit_forwards"
                ]
                for row in round_history
            )
        ),
        "total_fresh_quantum_backward_passes": (
            total_fresh_quantum_backward_passes
        ),
        "total_lazy_quantum_steps": (
            total_lazy_quantum_steps
        ),
        "overall_refresh_ratio": float(
            total_fresh_quantum_backward_passes
            / max(
                total_quantum_steps,
                1,
            )
        ),
        "qotph_runtime": {
            key: float(
                sum(
                    row[
                        "qotph_runtime"
                    ].get(
                        key,
                        0.0,
                    )
                    for row in round_history
                )
            )
            for key in sorted(
                set().union(
                    *[
                        row[
                            "qotph_runtime"
                        ].keys()
                        for row in round_history
                    ]
                )
                if round_history
                else set()
            )
        },
    }

    configuration = {
        "run_profile": RUN_PROFILE,
        "run_name": RUN_NAME,
        "seed": seed,
        "dataset": dataset,
        "device": str(
            DEVICE
        ),
        "model_variant": MODEL_VARIANT,
        "latent_dim": LATENT_DIM,
        "n_qubits": N_QUBITS,
        "k_circuits": K_CIRCUITS,
        "r_unique": R_UNIQUE,
        "ru_layers": RU_LAYERS,
        "qotph_mode": QOTPH_MODE,
        "qotph_execution": (
            QOTPH_EXECUTION
        ),
        "qotph_shots": QOTPH_SHOTS,
        "qotph_key_seed": (
            QOTPH_KEY_SEED
        ),
        "qotph_key_mode": QOTPH_KEY_MODE,
        "qotph_wire_order": (
            list(
                QOTPH_WIRE_ORDER
            )
        ),
        "quantum_training": (
            QUANTUM_TRAINING
        ),
        "tau_drift": TAU_DRIFT,
        "refresh_every": (
            REFRESH_EVERY
        ),
        "beta_qgrad": BETA_QGRAD,
        "gamma_stale": GAMMA_STALE,
        "warmup_fresh_steps": (
            WARMUP_FRESH_STEPS
        ),
        "minimum_refresh_ratio": (
            MIN_REFRESH_RATIO
        ),
        "maximum_refresh_ratio": (
            MAX_REFRESH_RATIO
        ),
        "maximum_stale_steps": (
            MAX_STALE_STEPS
        ),
        "adaptive_weights": {
            "error": (
                ADAPT_ERROR_WEIGHT
            ),
            "stagnation": (
                ADAPT_STAGNATION_WEIGHT
            ),
            "drift": (
                ADAPT_DRIFT_WEIGHT
            ),
            "uncertainty": (
                ADAPT_UNCERTAINTY_WEIGHT
            ),
            "fairness": (
                ADAPT_FAIRNESS_WEIGHT
            ),
        },
        "resource_aware": (
            USE_RESOURCE_AWARE_POLICY
        ),
        "circuit_masking": (
            USE_CIRCUIT_MASKING
        ),
        "bandwidth_sparse_upload": (
            USE_BANDWIDTH_SPARSE_UPLOAD
        ),
        "circuit_dropout_probability": (
            CIRCUIT_DROPOUT_PROB
        ),
        "partition_mode": (
            PARTITION_MODE
        ),
        "dirichlet_alpha": (
            DIRICHLET_ALPHA
        ),
        "balanced_samples_per_class_per_client": (
            BALANCED_SAMPLES_PER_CLASS_PER_CLIENT
        ),
        "balanced_val_per_class_per_client": (
            BALANCED_VAL_PER_CLASS_PER_CLIENT
        ),
        "number_clients": (
            number_clients
        ),
        "fraction_fit": frac_fit,
        "minimum_fit_clients": (
            min_fit_clients
        ),
        "rounds": rounds,
        "local_epochs_base": (
            max_epochs
        ),
        "batch_size": batch_size,
        "aggregation_mode": (
            AGGREGATION_MODE
        ),
        "tau_aggregation": TAU_AGG,
        "quality_ema": (
            USE_QUALITY_EMA
        ),
        "quality_ema_rho": (
            QUALITY_EMA_RHO
        ),
        "fedprox": USE_FEDPROX,
        "fedprox_mu": FEDPROX_MU,
        "server_ema": (
            USE_SERVER_EMA
        ),
        "server_ema_beta_classical": (
            SERVER_EMA_BETA_CLASSICAL
        ),
        "server_ema_beta_quantum": (
            SERVER_EMA_BETA_QUANTUM
        ),
        "layer_sparing": (
            USE_LAYER_SPARING
        ),
        "importance_ema_alpha": (
            IMPORTANCE_EMA_ALPHA
        ),
        "freeze_threshold": (
            FREEZE_THRESHOLD
        ),
        "freeze_warmup_rounds": (
            FREEZE_WARMUP_ROUNDS
        ),
        "freeze_patience": (
            FREEZE_PATIENCE
        ),
        "quality_dp": use_dp,
        "dp_epsilon_per_release": (
            DP_EPSILON
        ),
        "update_dp": (
            USE_UPDATE_DP
        ),
        "update_dp_clip_norm": (
            UPDATE_DP_CLIP_NORM
        ),
        "update_dp_noise_multiplier": (
            UPDATE_DP_NOISE_MULTIPLIER
        ),
        "selective_ckks": he,
        "encrypted_weight": (
            ENCRYPTED_FINAL_WEIGHT
        ),
        "lr_classical": (
            LR_CLASSICAL
        ),
        "lr_quantum": LR_QUANTUM,
    }

    checkpoint = {
        "model_state_dict": (
            clone_state_dict(
                global_model.state_dict()
            )
        ),
        "broadcast_state": (
            broadcast_state
        ),
        "previous_raw_state": (
            previous_raw_state
        ),
        "server_ema_state": (
            server_ema_state
        ),
        "round_history": (
            round_history
        ),
        "partition_stats": (
            partition_stats
        ),
        "resource_profiles": {
            cid: asdict(profile)
            for cid, profile in (
                resource_profiles.items()
            )
        },
        "adaptive_client_states": {
            cid: asdict(state)
            for cid, state in (
                adaptive_client_states.items()
            )
        },
        "qotph_unit_test_results": (
            qotph_unit_test_results
        ),
        "runtime_summary": (
            runtime_summary
        ),
        "configuration": (
            configuration
        ),
    }

    torch.save(
        checkpoint,
        model_save,
    )

    history_path = os.path.join(
        save_results,
        f"{RUN_NAME}_history.pkl",
    )
    with open(
        history_path,
        "wb",
    ) as history_file:
        pickle.dump(
            round_history,
            history_file,
        )

    result_row = {
        "run_name": RUN_NAME,
        "qotph_mode": QOTPH_MODE,
        "qotph_execution": (
            QOTPH_EXECUTION
        ),
        "quantum_training": (
            QUANTUM_TRAINING
        ),
        "resource_aware": (
            USE_RESOURCE_AWARE_POLICY
        ),
        "quality_dp": use_dp,
        "update_dp": (
            USE_UPDATE_DP
        ),
        "ckks": he,
        "final_accuracy": float(
            round_history[-1][
                "test_acc"
            ]
        ),
        "final_loss": float(
            round_history[-1][
                "test_loss"
            ]
        ),
        "runtime_seconds": (
            total_runtime_seconds
        ),
        "overall_refresh_ratio": (
            runtime_summary[
                "overall_refresh_ratio"
            ]
        ),
        "total_upload_mb": float(
            sum(
                row[
                    "actual_upload_bytes"
                ]
                for row in round_history
            )
            / 1e6
        ),
    }

    result_row_path = os.path.join(
        save_results,
        f"{RUN_NAME}_result_row.csv",
    )
    pd.DataFrame(
        [result_row]
    ).to_csv(
        result_row_path,
        index=False,
    )

    print(
        f"Saved checkpoint: {model_save}"
    )
    print(
        f"Saved round history: {history_path}"
    )
    print(
        f"Saved Stage-2 result row: "
        f"{result_row_path}"
    )


NameError: name 'save_results' is not defined

In [14]:
plain_features = model.quantum_features_plain(
    latent
)

protected_features = model.quantum_features_qotph(
    latent,
    mode="algorithm2",
    execution="analytic",
)

max_error = (
    plain_features - protected_features
).abs().max()

mean_error = (
    plain_features - protected_features
).abs().mean()

print("Maximum QOTPH error:", max_error.item())
print("Mean QOTPH error:", mean_error.item())

NameError: name 'model' is not defined

# Recommended Stage-2 ablation sequence

Run the same frozen architecture in this order:

1. `QOTPH_MODE="off"` with standard or fixed LazyQ.
2. `QOTPH_MODE="algorithm1"` with analytic execution.
3. `QOTPH_MODE="algorithm2"` with analytic execution.
4. Algorithm 2 + fixed LazyQ.
5. Algorithm 2 + adaptive LazyQ.
6. Algorithm 2 + adaptive LazyQ + resource-aware policy.
7. Add CKKS.
8. Add DP.
9. Run the full protected stack.
10. Repeat primary comparisons over at least three matched seeds.

Use `qotph_counts_smoke` before any large finite-shot run. A full finite-shot parameter-shift experiment can require orders of magnitude more circuit evaluations than analytic training.
